<div align="center">

# Hafta 5: Zaman serisi analizi ve EKG yapay zekâsı

**Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı**<br>
Türk Kardiyoloji Derneği · Prof. Dr. Utku Köse

[Hafta sayfası](README.md) · [Ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html) · [Etkileşimli lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/lab.html) · [PDF ders notu](Hafta05_Ders_Notlari.pdf) · [Colab'da aç](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-05/NB05_zaman_serisi_ve_ekg_yapay_zekasi.ipynb)

</div>

**Bu defter iki kısımdan oluşur.** Birincisinde kodu bir yapay zekâ (YZ) asistanı yazar; asistana ne yapacağını yazılı bir talimat, yani bir prompt (istem) anlatır. İkincisinde kodu Python ile kendiniz yazarsınız. İki kısım da aynı verilerle aynı sonuçlara varır; ders sayfasındaki her uygulama durağında bunlardan birini seçin.

| | Kimler için? | Ne yapılır? |
|---|---|---|
| **[Kısım 1: Prompt yolu](#part-1)** | Kod yazmayan katılımcılar | Kodu, hazır promptlarla bir YZ asistanı yazar; siz çalıştırır ve sonucu denetlersiniz. |
| **[Kısım 2: Python yolu](#part-2)** | Python öğrenmek isteyen katılımcılar | Kodu alıştırmalarla adım adım kendiniz yazar ve çalıştırırsınız. |

**Nerede çalışır?** Aynı defter, hiçbir kurulum gerektirmeyen Google Colab'da da kendi bilgisayarınızdaki Jupyter'de de (Anaconda; ders klasöründeki SETUP.tr.md dosyasına bakın) çalışır. Hangisini tercih ederseniz onu kullanın. Colab'da önce *Dosya → Drive'a kopya kaydet* (*File → Save a copy in Drive*) komutuyla kendi kopyanızı kaydedin; aksi hâlde sekme kapanınca yaptığınız değişiklikler kaybolur. Evrişimli ağı eğitmek sıradan bir dizüstü bilgisayarda yaklaşık bir dakika, tekrarlayan ağı eğitmek ise yaklaşık yarım dakika sürer; ekran kartı gerekmez. Hangi kısmı seçerseniz seçin önce 0. bölümü çalıştırın. Bu bölüm ders dosyalarını ve paketleri hazırlar.

**Veri ve mahremiyet.** Buradaki bütün veri setleri ya sentetiktir ya da açık bir araştırma veri tabanından gelmektedir. Gerçek hasta verisi yalnızca kendi bilgisayarınızda, kurumunuzun kurallarına uygun olarak ve etik kurul onayıyla işlenir; bulut tabanlı bir deftere ya da bir YZ sohbetine asla konmaz.

## 0. Kurulum

İlk hücre ders klasörünü bulur. Defter tek başına açıldıysa, örneğin Google Colab'da, `vclinic` ders paketini ve veri dosyalarını GitHub'dan indirir. İkinci hücre paketleri yükler, mesajların dilini seçer ve haftanın isteğe bağlı paketlerini denetler. İkinci hücre eğitimin 12 derivasyonlu EKG'lerini PhysioNet'ten indirir: PTB-XL'in 1., 2., 9. ve 10. katlarındaki 100 Hz'lik kayıtlar, yani 8.737 kayıt ve iki tablo, toplam yaklaşık 216 MB. İlk çalıştırma birkaç dakika sürer; sonraki çalıştırmalarda zaten var olan dosyalar atlanır. PTB-XL, Creative Commons Atıf 4.0 Uluslararası Lisansı (CC BY 4.0) ile yayımlanmaktadır. Her kullanımda Wagner ve arkadaşlarının Scientific Data dergisindeki 2020 tarihli makalesine ve PhysioNet'teki veri setine atıf yapılır.

In [ ]:
#@title Ders dosyaları: Bu hücreyi çalıştırın (açmanıza gerek yok)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # İndirilmiş bir kopyada, örneğin Colab'da, paketin dosyaları yenilenir.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Ders dosyaları şu klasöre indirildi:", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Kurulum: paketler, dil ve çizim biçimi
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import copy
import math
from collections import Counter

from sklearn.metrics import roc_auc_score, roc_curve
from vclinic import ptbxl, beats, holter

try:
    import torch
    import torch.nn as nn
except ImportError as error:
    raise ImportError("PyTorch kurulu değil. SETUP.tr.md dosyasında anlatıldığı gibi kurun, ardından çekirdeği yeniden başlatın.") from error
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 6, 0):
    raise ImportError("vclinic ders paketi bu defterden daha eskidir. Ders klasörünü güncelleyin (git pull komutuyla ya da ZIP dosyasını yeniden indirerek) ve çekirdeği yeniden başlatın.")
teach.set_language("tr")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026
torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))
print("PyTorch", torch.__version__)
ptbxl.download()
print("vclinic sürümü", vclinic.__version__)
print("Ders klasörü:", ROOT)
for name in ["ipywidgets"]:
    try:
        importlib.import_module(name)
        print(f"{name}: kurulu")
    except ImportError:
        print(f"{name}: kurulu değil (isteğe bağlı; defter yerleşik bir yedek yol kullanır)")

In [ ]:
#@title Alıştırmaların beklenen cevapları (bu hücreyi çalıştırın; açmayı alıştırmaları denedikten sonraya bırakın)
teach.register_answers({
    "Alıştırma 5.1": (4, 0),
    "Alıştırma 5.2": (4, 0),
    "Alıştırma 5.3": (1.168, 0.001),
    "Alıştırma A": (60, 0),
    "Alıştırma B": (0.704, 0.001),
    "Alıştırma C": (60396, 0),
    "Alıştırma D": (0.703, 0.03),
    "Alıştırma E": (3558, 0),
    "Alıştırma F": (1.168, 0.001),
    "Alıştırma G": (215, 0),
    "Alıştırma H": (217, 0),
    "Alıştırma I": (14.7, 0.05),
    "Alıştırma J": (0.882, 0.001),
    "Adım 1.1": (0.914, 0.001),
    "Adım 1.2": (0.881, 0.03),
    "Adım 1.3": (0.975, 0.02),
    "Adım 1.4": (0.636, 0.06),
    "Adım 2.1": (19, 0),
    "Adım 2.2": (9, 0),
    "Adım 2.3": (51, 0),
    "Adım 3.1": (519, 0),
    "Adım 3.2": (18.0, 0.05),
    "Adım 3.3": (75, 0),
})
print("Beklenen cevaplar hazır.")

In [ ]:
#@title Alıştırma promptlarının tabloları (bu hücreyi çalıştırın; Python yolunda okunması gerekmez)
def alistirma_tablosu(anahtar):
    """Bir alıştırma promptunun "veri" tablosu; Python yolunun verilerinden kurulur."""
    g = globals()
    k = anahtar
    leads = list(beats.LEADS)
    if k == "5.1":
        return pd.concat([pd.DataFrame({"zaman": np.round(np.arange(1000) / 100, 2)}), pd.DataFrame(g["x"].T, columns=leads)], axis=1)
    if k == "5.2":
        return g["labels_vc"].reset_index().rename(columns={"patient_id": "hasta"})
    if k == "5.3":
        return pd.DataFrame({"hasta": g["labels_vc"].index, "D2 derivasyonu": list(g["X"][:, 1])})
    if k == "A":
        return pd.concat([pd.DataFrame({"kayıt": g["ecgs"]["ecg_id"].to_numpy()[g["te"]]}),
                          g["labels"].iloc[np.where(g["te"])[0]].reset_index(drop=True)], axis=1)
    if k in ("B", "D", "F", "G", "H", "I", "J"):
        return g["_exercise_tables"][k]()
    raise KeyError(k)


def _st_rows(mask):
    ix = np.where(mask)[0]
    st = st_table.set_index("ecg_id").loc[ecgs["ecg_id"].to_numpy()[ix]]
    return pd.concat([pd.DataFrame({"kayıt": ecgs["ecg_id"].to_numpy()[ix],
                                    "cinsiyet": np.where(ecgs["sex"].to_numpy()[ix] == "M", "E", "K"),
                                    "yaş": ecgs["age"].to_numpy()[ix].astype(int),
                                    "akut enfarktüs": ecgs["acute_mi"].to_numpy()[ix].astype(int)}),
                      st[leads_order].reset_index(drop=True).round(3)], axis=1)


leads_order = list(beats.LEADS)
_exercise_tables = {
    "B": lambda: pd.DataFrame({"kayıt": ecgs["ecg_id"].to_numpy()[te], "RR": [np.round(rr_list[k], 2) for k in np.where(te)[0]],
                               "AF": labels["AFIB"].to_numpy()[te]}),
    "D": lambda: pd.concat([pd.DataFrame({"kayıt": ecgs["ecg_id"].to_numpy()[te]}), labels.iloc[np.where(te)[0]].reset_index(drop=True),
                            pd.DataFrame(np.round(test_prob, 4), columns=[f"olasılık {c}" for c in RHYTHMS])], axis=1),
    "F": lambda: pd.concat([pd.DataFrame({"zaman": np.arange(-30, 56) * 10}), pd.DataFrame(np.round(case_beat.T, 3), columns=leads_order)], axis=1),
    "G": lambda: _st_rows(va | te),
    "H": lambda: pd.DataFrame({"kayıt": qt_table["ecg_id"], "QT (ms)": qt_table["qt_ms"].round(2), "RR (s)": qt_table["rr_s"].round(4)}),
    "I": lambda: pd.DataFrame({"zaman": np.round(holter_beats["time"], 3), "RR": np.round(holter_beats["RR"], 3)}),
    "J": lambda: pd.DataFrame({"kayıt": ecgs["ecg_id"].to_numpy(), "hasta": ecgs["patient_id"].to_numpy().astype(int),
                               "küme": [{"training": "eğitim", "validation": "doğrulama", "test": "test"}[s] for s in split],
                               "RR": [np.round(r, 2) for r in rr_list], "AF": labels["AFIB"].to_numpy()}),
}
print("Alıştırma promptlarının tabloları hazır.")

<a id="part-1"></a>

# Kısım 1: Prompt yolu

**Kimler için?** Bu kısım kod yazmayan katılımcılar içindir. Burada hiç kod yazmazsınız. Kodu, hazır bir prompttan, yani yazılı bir talimattan yararlanarak bir yapay zekâ (YZ) asistanı yazar; siz kodu çalıştırır ve sonucu denetlersiniz.

**Neye ihtiyacınız var?** Python kodu yazabilen bir YZ asistanı yeterlidir. Google Colab'ın içinde Gemini bulunur; pencerenin üstündeki Gemini düğmesiyle açılır. Başka bir tarayıcı sekmesinde açılan ChatGPT, Claude, Copilot ya da Gemini de aynı işi görür.

**Bir adım nasıl ilerler?**

1. Adımın amacını okuyun.
2. Promptun altındaki veri hücresini çalıştırın. Hücre, promptun tarif ettiği "veri" tablosunu hazırlar ve tablonun ilk ve son satırlarını gösterir.
3. Promptu kopyalayın: Fareyle seçin ve Ctrl+C'ye (Mac'te Cmd+C) basın.
4. Promptu YZ asistanına yapıştırın ve gönderin.
5. Asistanın verdiği kodu kopyalayın, veri hücresinin altındaki hücreye `#` ile başlayan satırın yerine yapıştırın ve hücreyi ▶ düğmesiyle ya da Shift+Enter ile çalıştırın.
6. Hemen altındaki kontrol hücresini çalıştırın. Hücre *doğru ✓* ya da *henüz değil ✗* yazar.
7. *Henüz değil* yazarsa aşağıdaki onarım promptunu kodla birlikte asistana gönderin. Sonuç yine yanlışsa *Hazır kod* hücresini çalıştırın. Bu hücre her zaman doğru sonucu verir; böylece bir sonraki adıma geçebilirsiniz.

**"veri" ve "cevap".** 1. haftada olduğu gibi her adımın veri hücresi, promptun tarif ettiği "veri" tablosunu hazırlar. Asistan sonucu "cevap" adıyla saklar; kontrol hücresi de sonucu bu addan okur.

**Her durağın başında** o durağın hazırlık hücresini bir kez çalıştırın. Hücre durağın verilerini yükler; böylece bir durağa başka bir gün de başlanabilir.

**Onarım promptu** (çalışan ama kontrolden geçmeyen kod için):

> Aşağıdaki kod şu görev için yazıldı: [promptun Görev satırını yapıştırın]. Kod çalışıyor, ancak dersin kontrolü "henüz değil" yazıyor. Hatayı bul, iki cümleyle açıkla ve düzeltilmiş kodu ver. Sonuç yine "cevap" adıyla saklanmalı.  
>
> [kodu buraya yapıştırın]  

**Hata mesajı için prompt** (hata vererek duran kod için):

> Aşağıdaki kod bir Jupyter defterinde çalıştırıldığında altta verilen hata mesajını veriyor. Hatanın nedenini sade bir dille açıkla ve düzeltilmiş kodu ver.  
>
> [kodu buraya yapıştırın]  
>
> [hata mesajını buraya yapıştırın]  

Haftanın bütün promptları [prompt kütüphanesinde](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-05/PROMPTLAR.md) de yer alır; GitHub her promptun yanında bir kopyalama düğmesi gösterir.

**Hiç değişmeyen kural.** Promptlarda yalnızca dersin sentetik ya da açık verileri bulunur. Bir YZ asistanına gerçek hasta verisi asla yapıştırılmaz.

## Durak 1: Ritim: Altı etiket için bir kural ve bir ağ

*Ders: [1. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html#stop-1)* · *Python yolunda aynı sonuç: Kısım 2, 1-5. bölümler*

Bu durak oturumun mini projesidir. PTB-XL'in 12 derivasyonlu EKG'leri önce bir kardiyoloğun ritmi okuduğu gibi RR aralıklarından, ardından altı etiketi aynı anda öğrenen tek boyutlu evrişimli bir ağ tarafından okunur. Ağ 1. ve 2. katlarla eğitilir, eşikleri 9. katta seçilir; kural da ağ da 10. katta değerlendirilir.

**Durağın sonunda:** Kuralın AF için duyarlılığını (0,914), ağın 9. kattaki ortalama AUC değerini (yaklaşık 0,881), ağın 10. katta AF için AUC değerini (yaklaşık 0,975) ve 9. katta seçilen eşikle ağın AF alarmının pozitif öngörü değerini (yaklaşık 0,636) bulacaksınız.

In [ ]:
#@title 1. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
if "ecgs" not in globals():
    ecgs = ptbxl.table()
    signals = ptbxl.signals(ecgs)
    RHYTHMS = ["SR", "AFIB", "STACH", "SBRAD", "SARRH", "PVC"]
    labels = ptbxl.labels(ecgs, RHYTHMS)
    split = ecgs["split"].to_numpy()
    tr, va, te = split == "training", split == "validation", split == "test"

    r_peak_list = []
    for start in range(0, len(signals), 500):
        r_peak_list += beats.r_peaks_many(signals[start:start + 500])
    rr_list = [np.diff(p) / 100 for p in r_peak_list]
    cv = np.array([rr.std(ddof=1) / rr.mean() if len(rr) >= 3 else np.nan for rr in rr_list])


def macro_auc(y_true, prob):
    """Etiketlerin AUC değerlerinin ortalaması: Her sütun için bir AUC."""
    return float(np.mean([roc_auc_score(y_true[:, k], prob[:, k]) for k in range(y_true.shape[1])]))


def youden_threshold(y_true, prob):
    """Duyarlılık + özgüllük toplamının en yüksek olduğu eşik."""
    fpr, tpr, thresholds = roc_curve(y_true, prob)
    return float(thresholds[np.argmax(tpr - fpr)])


def se_sp_ppv(y_true, alarm):
    """Evet ya da hayır biçimindeki bir kararın duyarlılığı, özgüllüğü ve pozitif öngörü değeri."""
    y_true, alarm = np.asarray(y_true, dtype=bool), np.asarray(alarm, dtype=bool)
    tp, fp = int((alarm & y_true).sum()), int((alarm & ~y_true).sum())
    return tp / y_true.sum(), 1 - fp / (~y_true).sum(), tp / max(1, tp + fp)


def conv_block(c_in, c_out, k):
    """Bir katman: Zaman boyunca adımı 2 olan bir evrişim, batch normalizasyonu ve ReLU."""
    return [nn.Conv1d(c_in, c_out, k, stride=2, padding=k // 2), nn.BatchNorm1d(c_out), nn.ReLU()]


class RhythmNet(nn.Module):
    """12 derivasyon üzerinde dört evrişim, her kanalın zaman boyunca ortalaması ve her etiket için bir çıktı."""

    def __init__(self, n_labels=6):
        super().__init__()
        self.features = nn.Sequential(*conv_block(12, 32, 7), *conv_block(32, 64, 5),
                                      *conv_block(64, 64, 5), *conv_block(64, 128, 3))
        self.out = nn.Linear(128, n_labels)

    def forward(self, x):
        return self.out(self.features(x).mean(dim=2))


def predict(net, *inputs, batch=256):
    """Etiketlerin olasılıkları; eğitim yapılmadan batch'ler hâlinde hesaplanır."""
    net.eval()
    with torch.no_grad():
        return torch.cat([torch.sigmoid(net(*[x[i:i + batch] for x in inputs]))
                          for i in range(0, len(inputs[0]), batch)]).numpy()


def _course_rhythmnet():
    """Dersin ağı; bir kez eğitilir (yaklaşık bir dakika) ve 9. ile 10. katlarda altı etiketin olasılıklarını verir."""
    if "_rhythm_cache" in globals():
        return _rhythm_cache
    global x_all, y_all, x_train, y_train, x_val, x_test, y_val, y_test
    x_all = torch.tensor(np.clip(signals, -5, 5))
    y_all = torch.tensor(np.array(labels, dtype=np.float32))
    x_train, y_train = x_all[tr], y_all[tr]
    x_val, x_test = x_all[va], x_all[te]
    y_val, y_test = labels.to_numpy()[va], labels.to_numpy()[te]

    torch.manual_seed(RANDOM_STATE)
    rhythm_net = RhythmNet(len(RHYTHMS))
    EPOCHS, BATCH = 15, 64
    optimizer = torch.optim.Adam(rhythm_net.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.003, total_steps=EPOCHS * math.ceil(len(x_train) / BATCH))
    loss_fn = nn.BCEWithLogitsLoss()
    best_auc, best_epoch, val_history = -1.0, 0, []
    for epoch in range(1, EPOCHS + 1):
        rhythm_net.train()
        order = torch.randperm(len(x_train))
        for i in range(0, len(order), BATCH):
            b = order[i:i + BATCH]
            optimizer.zero_grad()
            loss = loss_fn(rhythm_net(x_train[b]), y_train[b])
            loss.backward()
            optimizer.step()
            scheduler.step()
        val_auc = macro_auc(y_val, predict(rhythm_net, x_val))
        val_history.append(val_auc)
        if val_auc > best_auc:
            best_auc, best_epoch, best_state = val_auc, epoch, copy.deepcopy(rhythm_net.state_dict())
        print(f"epoch {epoch:2d} · 9. katta ortalama AUC: {val_auc:.3f}")
    rhythm_net.load_state_dict(best_state)
    val_prob, test_prob = predict(rhythm_net, x_val), predict(rhythm_net, x_test)
    globals()["_rhythm_cache"] = (val_prob, test_prob)
    return _rhythm_cache

_SETS = {"training": "eğitim", "validation": "doğrulama", "test": "test"}
print(len(ecgs), "kayıt;", {s: int((split == s).sum()) for s in ("training", "validation", "test")})
print("1. durak hazır.")

### Adım 1.1: RR aralıklarından bir kural

**Amaç.** Bu adımda AF için klinik kuralın ilk yarısı, yani düzensiz biçimde düzensiz RR aralıkları, test katında sınanır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: RR aralıklarına dayanan basit bir kuralın 10 saniyelik 12 derivasyonlu EKG'lerde atriyal fibrilasyonu (AF) ne kadar iyi bulduğunu öğrenmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; klinik ortamda alınmış 10 saniyelik 12 derivasyonlu EKG'lerden oluşan herkese açık PTB-XL veri setinin test katındaki 2.198 kaydı tutar. Her satır bir kayıttır: "kayıt" sütunu kaydın numarasını verir, "RR" sütunu kaydın 12 derivasyonun hepsinden bulunan R tepelerine göre hesaplanmış RR aralıklarını saniye olarak tutar, "AF" sütunu ise kardiyologlar atriyal fibrilasyon tanımladıysa 1, tanımlamadıysa 0'dır.  
> Görev: En az üç RR aralığı olan her kayıt için değişim katsayısını hesapla: Aralıkların standart sapması (paydada aralık sayısının bir eksiği kullanılır) ortalamalarına bölünür. Değişim katsayısı 0,10'un üstünde olan kayda AF de. Kuralı "AF" sütunuyla karşılaştır; duyarlılığını, özgüllüğünü ve pozitif öngörü değerini hesapla. AF'li ve AF'siz kayıtların değişim katsayılarını iki histogram olarak çiz ve 0,10 düzeyine dikey bir çizgi ekle.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Alarm sayısını ve üç değeri yazdır; duyarlılığı üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Alarmların sayısı doğru ve yanlış alarmların toplamına, duyarlılık da doğru alarmların AF'li kayıtların sayısına bölümüne eşit olmalı.  

In [ ]:
#@title Adım 1.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_i = np.where(te)[0]
veri = pd.DataFrame({"kayıt": ecgs["ecg_id"].to_numpy()[_i], "RR": [np.round(rr_list[k], 2) for k in _i],
                     "AF": labels["AFIB"].to_numpy()[_i]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.1 kontrolü
teach.check_step("Adım 1.1", "cevap", globals())

In [ ]:
#@title Adım 1.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_i = np.where(te)[0]
veri = pd.DataFrame({"kayıt": ecgs["ecg_id"].to_numpy()[_i], "RR": [np.round(rr_list[k], 2) for k in _i],
                     "AF": labels["AFIB"].to_numpy()[_i]})
dk = np.array([np.std(r, ddof=1) / np.mean(r) if len(r) >= 3 else np.nan for r in veri["RR"]])
alarm = dk > 0.10
af = veri["AF"].to_numpy() == 1
dp, yp = int((alarm & af).sum()), int((alarm & ~af).sum())
cevap = round(dp / af.sum(), 3)
ozgulluk = 1 - yp / (~af).sum()
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(dk[~af], bins=np.linspace(0, 0.5, 51), color="#1F5F8B", alpha=0.6, label="AF yok")
ax.hist(dk[af], bins=np.linspace(0, 0.5, 51), color="#9B1B1B", alpha=0.7, label="AF")
ax.axvline(0.10, color="k", ls="--", lw=1)
ax.set_xlabel("RR aralıklarının değişim katsayısı")
ax.set_ylabel("kayıt")
ax.legend()
plt.show()
print("Alarmlar:", int(alarm.sum()), "· duyarlılık", cevap, "· özgüllük", round(ozgulluk, 3),
      "· PPV", round(dp / max(1, dp + yp), 3))
teach.check_step("Adım 1.1", "cevap", globals())

**Sonuç ne gösteriyor?** Kural AF'li kayıtların yüzde 91 kadarını bulur, ama alarmlarının yarısından fazlası yanlıştır: Erken vurular ve sinüs aritmisi de RR aralıklarını düzensizleştirir. Histogramlar, hiçbir eşiğin ortadan kaldıramayacağı örtüşmeyi göstermektedir.

### Adım 1.2: Altı etiket için tek boyutlu bir ağ

**Amaç.** Bu adımda 12 derivasyonun sinyalinin tamamını okuyan ve aynı anda altı etiket veren bir ağ eğitilir. Eğitim yaklaşık bir dakika sürer.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: 10 saniyelik 12 derivasyonlu EKG'leri okuyan ve altı ritim bulgusunu aynı anda tanıyan bir ağ eğitmek ve bu ağın bulguları yeni kayıtlarda ne kadar iyi ayırdığını öğrenmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; klinik ortamda alınmış 10 saniyelik 12 derivasyonlu EKG'lerden oluşan herkese açık PTB-XL veri setinden 6.539 kayıt tutar. Her satır bir kayıttır: "küme" sütunu kaydın kümesini verir ("eğitim": 1. ve 2. katlar, 4.356 kayıt; "doğrulama": 9. kat, 2.183 kayıt). "EKG" sütununun her hücresi kaydı, 100 Hz'de alınmış 12 derivasyon × 1.000 örnekten oluşan ve mV cinsinden değerler içeren bir dizi olarak tutar; derivasyonların sırası D1, D2, D3, aVR, aVL, aVF ve V1-V6'dır. "SR", "AFIB", "STACH", "SBRAD", "SARRH" ve "PVC" adlı altı sütun, kardiyologlar ilgili ifadeyi verdiyse 1, vermediyse 0'dır. Bu ifadeler sırasıyla sinüs ritmi, atriyal fibrilasyon, sinüs taşikardisi, sinüs bradikardisi, sinüs aritmisi ve ventriküler erken vurulardır.  
> Görev: Altı etiketin her birine bir skor veren tek boyutlu bir evrişimli ağ kur. Ağda zaman boyunca dört evrişim katmanı bulunur; her katmanın adımı 2'dir ve her katmanı batch normalizasyonu ile ReLU izler. Katmanlar sırasıyla 12 derivasyon üzerinde 7 örneklik 32 çekirdek, ardından 5 örneklik 64 çekirdek, 5 örneklik 64 çekirdek ve 3 örneklik 128 çekirdekten oluşur. 128 kanalın her birinin zaman boyunca ortalaması, altı çıktısı olan son bir doğrusal katmana verilir. Eğitimden önce sinyallerin değerlerini −5 ile 5 mV aralığıyla sınırla. Ağı eğitim kayıtlarıyla, karıştırılmış 64 kayıtlık gruplar hâlinde 15 epoch boyunca eğit; kayıp olarak altı etiketin ikili çapraz entropisini, optimizasyon algoritması olarak Adam'ı kullan. Tek döngülü (one-cycle) bir öğrenme hızı kullan ve en yüksek değerini 0,003 yap. Her epoch'tan sonra doğrulama kayıtlarında her etiketin ROC eğrisi altındaki alanını (AUC) ve bunların ortalamasını hesapla; ortalamanın en yüksek olduğu epoch'un ağırlıklarını sakla. Rastgele seçimler her çalıştırmada aynı olsun; bunun için başlangıç değeri olarak 2026 sayısını kullan.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Parametre sayısını, her epoch'un ortalama AUC değerini ve en iyi epoch'u yazdır; en iyi ortalama AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Ağın 59.622 parametresi olmalı ve ortalama AUC 0,5 ile 1 arasında olmalı.  

In [ ]:
#@title Adım 1.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_i = np.where(tr | va)[0]
veri = pd.concat([pd.DataFrame({"küme": [_SETS[s] for s in split[_i]], "EKG": list(signals[_i])}),
                  labels.iloc[_i].reset_index(drop=True)], axis=1)
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.2 kontrolü
teach.check_step("Adım 1.2", "cevap", globals())

In [ ]:
#@title Adım 1.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_i = np.where(tr | va)[0]
veri = pd.concat([pd.DataFrame({"küme": [_SETS[s] for s in split[_i]], "EKG": list(signals[_i])}),
                  labels.iloc[_i].reset_index(drop=True)], axis=1)


def conv_block(c_in, c_out, k):
    """Bir katman: Zaman boyunca adımı 2 olan bir evrişim, batch normalizasyonu ve ReLU."""
    return [nn.Conv1d(c_in, c_out, k, stride=2, padding=k // 2), nn.BatchNorm1d(c_out), nn.ReLU()]


class RhythmNet(nn.Module):
    """12 derivasyon üzerinde dört evrişim, her kanalın zaman boyunca ortalaması ve her etiket için bir çıktı."""

    def __init__(self, n_labels=6):
        super().__init__()
        self.features = nn.Sequential(*conv_block(12, 32, 7), *conv_block(32, 64, 5),
                                      *conv_block(64, 64, 5), *conv_block(64, 128, 3))
        self.out = nn.Linear(128, n_labels)

    def forward(self, x):
        return self.out(self.features(x).mean(dim=2))


def predict(net, *inputs, batch=256):
    """Etiketlerin olasılıkları; eğitim yapılmadan batch'ler hâlinde hesaplanır."""
    net.eval()
    with torch.no_grad():
        return torch.cat([torch.sigmoid(net(*[x[i:i + batch] for x in inputs]))
                          for i in range(0, len(inputs[0]), batch)]).numpy()

egitim = np.array(veri["küme"] == "eğitim")
x = torch.tensor(np.clip(np.stack(veri["EKG"]), -5, 5).astype(np.float32))
y = torch.tensor(np.array(veri[RHYTHMS], dtype=np.float32))
x_egitim, y_egitim = x[egitim], y[egitim]
x_dogrulama, y_dogrulama = x[~egitim], y[~egitim].numpy()
torch.manual_seed(2026)
ag = RhythmNet(6)
print("Parametreler:", sum(p.numel() for p in ag.parameters()))
optimizer = torch.optim.Adam(ag.parameters(), lr=0.001)
program = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.003, total_steps=15 * math.ceil(len(x_egitim) / 64))
kayip = nn.BCEWithLogitsLoss()
en_iyi, en_iyi_epoch = -1.0, 0
for epoch in range(1, 16):
    ag.train()
    sira = torch.randperm(len(x_egitim))
    for bas in range(0, len(sira), 64):
        g = sira[bas:bas + 64]
        optimizer.zero_grad()
        kayip(ag(x_egitim[g]), y_egitim[g]).backward()
        optimizer.step()
        program.step()
    olasilik = predict(ag, x_dogrulama)
    ortalama_auc = float(np.mean([roc_auc_score(y_dogrulama[:, k], olasilik[:, k]) for k in range(6)]))
    if ortalama_auc > en_iyi:
        en_iyi, en_iyi_epoch = ortalama_auc, epoch
    print("epoch", epoch, "ortalama AUC:", round(ortalama_auc, 3))
cevap = round(en_iyi, 3)
print("En iyi epoch:", en_iyi_epoch, "· ortalama AUC:", cevap)
teach.check_step("Adım 1.2", "cevap", globals())

**Sonuç ne gösteriyor?** Dersin çalıştırmasında 9. kattaki ortalama AUC, ilk epoch'tan sonraki yaklaşık 0,65 değerinden yaklaşık 0,881 değerine yükselmiştir. Sayı bir bilgisayardan diğerine biraz değişir; çünkü grupların sırası ve işlemcilerin aritmetiği farklıdır.

### Adım 1.3: Test katında her etiket

**Amaç.** Bu adımda dersin ağı, hiç görmediği kayıtlarda etiket etiket değerlendirilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Altı ritim bulgusunun her biri için bir ağın, bulguyu taşıyan 12 derivasyonlu EKG'leri taşımayanlardan ne kadar iyi ayırdığını öğrenmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; PTB-XL'in test katındaki 2.198 kaydı tutar. Her satır bir kayıttır: "kayıt" sütunu kaydın numarasını verir. "SR", "AFIB", "STACH", "SBRAD", "SARRH" ve "PVC" adlı altı sütun, kardiyologlar ilgili ifadeyi verdiyse 1, vermediyse 0'dır. Bu ifadeler sırasıyla sinüs ritmi, atriyal fibrilasyon, sinüs taşikardisi, sinüs bradikardisi, sinüs aritmisi ve ventriküler erken vurulardır. "olasılık SR" ile "olasılık PVC" arasındaki altı sütun, 1. ve 2. katlarla eğitilmiş tek boyutlu evrişimli bir ağın altı etikete verdiği olasılıkları tutar.  
> Görev: Altı etiketin her biri için, o etiketin olasılığının 0 ve 1 değerlerinden oluşan kendi sütununa karşı ROC eğrisi altındaki alanını (AUC) hesapla. Altı ROC eğrisini tek bir şekilde çiz ve her birinin AUC değerini açıklamaya yaz.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Etiketi, o etiketi taşıyan kayıt sayısını ve AUC değerini gösteren küçük bir tablo yazdır; "AFIB" etiketinin AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her AUC 0 ile 1 arasında olmalı; bir kayıt birkaç etiket taşıyabildiği için kayıt sayılarının toplamı en az 2.198 olmalı.  

In [ ]:
#@title Adım 1.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_val_prob, _test_prob = _course_rhythmnet()
_i = np.where(te)[0]
veri = pd.concat([pd.DataFrame({"kayıt": ecgs["ecg_id"].to_numpy()[_i]}), labels.iloc[_i].reset_index(drop=True),
                  pd.DataFrame(np.round(_test_prob, 4), columns=[f"olasılık {c}" for c in RHYTHMS])], axis=1)
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.3 kontrolü
teach.check_step("Adım 1.3", "cevap", globals())

In [ ]:
#@title Adım 1.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_val_prob, _test_prob = _course_rhythmnet()
_i = np.where(te)[0]
veri = pd.concat([pd.DataFrame({"kayıt": ecgs["ecg_id"].to_numpy()[_i]}), labels.iloc[_i].reset_index(drop=True),
                  pd.DataFrame(np.round(_test_prob, 4), columns=[f"olasılık {c}" for c in RHYTHMS])], axis=1)
satirlar = []
fig, ax = plt.subplots(figsize=(4.6, 4.2))
for etiket in RHYTHMS:
    y, p = veri[etiket], veri[f"olasılık {etiket}"]
    auc = roc_auc_score(y, p)
    ypo, dpo, _ = roc_curve(y, p)
    ax.plot(ypo, dpo, lw=1.2, label=f"{etiket} {auc:.3f}")
    satirlar.append({"etiket": etiket, "kayıt": int(y.sum()), "AUC": round(auc, 3)})
ax.plot([0, 1], [0, 1], color="#999999", lw=0.8, ls="--")
ax.set_xlabel("1 − özgüllük")
ax.set_ylabel("duyarlılık")
ax.legend(fontsize=8)
plt.show()
print(pd.DataFrame(satirlar))
cevap = round(roc_auc_score(veri["AFIB"], veri["olasılık AFIB"]), 3)
teach.check_step("Adım 1.3", "cevap", globals())

**Sonuç ne gösteriyor?** Ağ AF'yi yaklaşık 0,975 AUC ile ayırır; bu değer, aynı kayıtlarda 0,939'a ulaşan 1.1. adımdaki kuraldan daha iyidir. En zayıf etiket sinüs aritmisidir: Bu etiket kalp hızının birkaç atım boyunca yavaşça değişmesine bağlıdır; ağ ise bunu yalnızca 10 saniye üzerinden aldığı ortalama aracılığıyla görür.

### Adım 1.4: 9. katta seçilen eşikler

**Amaç.** Bu adımda olasılıklar, her etiket için 9. katta seçilip 10. katta değerlendirilen bir eşikle kararlara dönüştürülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Bir ağın altı ritim bulgusu için verdiği olasılıkları, her bulgu için testten ayrı tutulan kayıtlarda seçilmiş bir eşikle kararlara dönüştürmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; PTB-XL'in 9. ve 10. katlarındaki 4.381 kaydı tutar. Her satır bir kayıttır: "kayıt" sütunu kaydın numarasını, "küme" sütunu ise kümesini verir ("doğrulama": 9. kat; "test": 10. kat). "SR", "AFIB", "STACH", "SBRAD", "SARRH" ve "PVC" adlı altı sütun, kardiyologlar ilgili ifadeyi verdiyse 1, vermediyse 0'dır. Bu ifadeler sırasıyla sinüs ritmi, atriyal fibrilasyon, sinüs taşikardisi, sinüs bradikardisi, sinüs aritmisi ve ventriküler erken vurulardır. "olasılık SR" ile "olasılık PVC" arasındaki altı sütun, 1. ve 2. katlarla eğitilmiş tek boyutlu evrişimli bir ağın altı etikete verdiği olasılıkları tutar.  
> Görev: Her etiket için doğrulama kayıtlarında, duyarlılık ile özgüllüğün toplamının en yüksek olduğu olasılık eşiğini seç. Her eşiği test kayıtlarına uygula: Bir etiketin olasılığı eşiğine ulaştığında o etiket verilir. Her etiketin test kayıtlarındaki duyarlılığını, özgüllüğünü ve pozitif öngörü değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Etiketi, eşiğini ve üç değeri gösteren bir tablo yazdır; "AFIB" etiketinin test kayıtlarındaki pozitif öngörü değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Eşikler yalnızca doğrulama kayıtlarından gelmeli; her duyarlılık ve özgüllük 0 ile 1 arasında olmalı.  

In [ ]:
#@title Adım 1.4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_val_prob, _test_prob = _course_rhythmnet()
_i = np.where(va | te)[0]
_prob = np.zeros((len(ecgs), len(RHYTHMS)))
_prob[va], _prob[te] = _val_prob, _test_prob
veri = pd.concat([pd.DataFrame({"kayıt": ecgs["ecg_id"].to_numpy()[_i], "küme": [_SETS[s] for s in split[_i]]}),
                  labels.iloc[_i].reset_index(drop=True),
                  pd.DataFrame(np.round(_prob[_i], 4), columns=[f"olasılık {c}" for c in RHYTHMS])], axis=1)
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.4 kontrolü
teach.check_step("Adım 1.4", "cevap", globals())

In [ ]:
#@title Adım 1.4 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_val_prob, _test_prob = _course_rhythmnet()
_i = np.where(va | te)[0]
_prob = np.zeros((len(ecgs), len(RHYTHMS)))
_prob[va], _prob[te] = _val_prob, _test_prob
veri = pd.concat([pd.DataFrame({"kayıt": ecgs["ecg_id"].to_numpy()[_i], "küme": [_SETS[s] for s in split[_i]]}),
                  labels.iloc[_i].reset_index(drop=True),
                  pd.DataFrame(np.round(_prob[_i], 4), columns=[f"olasılık {c}" for c in RHYTHMS])], axis=1)
dog = veri[veri["küme"] == "doğrulama"]
test = veri[veri["küme"] == "test"]
satirlar = []
for etiket in RHYTHMS:
    sutun = f"olasılık {etiket}"
    ypo, dpo, esikler = roc_curve(dog[etiket], dog[sutun])
    esik = float(esikler[np.argmax(dpo - ypo)])
    y = test[etiket].to_numpy() == 1
    verilen = test[sutun].to_numpy() >= esik
    dp, yp = int((verilen & y).sum()), int((verilen & ~y).sum())
    satirlar.append({"etiket": etiket, "eşik": round(esik, 3),
                     "duyarlılık": round(dp / y.sum(), 3), "özgüllük": round(1 - yp / (~y).sum(), 3),
                     "PPV": round(dp / max(1, dp + yp), 3)})
tablo = pd.DataFrame(satirlar)
print(tablo)
cevap = float(tablo.loc[tablo["etiket"] == "AFIB", "PPV"].iloc[0])
teach.check_step("Adım 1.4", "cevap", globals())

**Sonuç ne gösteriyor?** 9. katta seçilen eşikle ağın AF alarmı her 100 alarmın yaklaşık 64 kadarında doğrudur. Seyrek etiketlerin eşikleri 0,5'in çok altında kalır ve pozitif öngörü değerleri düşüktür: Kayıtların yüzde 3'ünün taşıdığı bir etiket, AUC değeri iyi olsa bile çok sayıda yanlış alarm üretir.

**1. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html#stop-1) dönün ve duraktan sonrasını okumaya devam edin.

## Durak 2: ST yükselmesi: Medyan atım, J noktası ve alarm

*Ders: [2. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html#stop-2)* · *Python yolunda aynı sonuç: Kısım 2, 6-7. bölümler*

Haftanın vakası olan 13913 numaralı kayıt, burada bir STEMI alarmının okuduğu gibi okunur. Tek tek atımlar R tepelerine göre hizalanır ve medyanları her derivasyon için temiz bir atım verir. J noktasındaki ST sapması PR segmentine göre ölçülür, alarmın kararını evrensel tanımın ölçütleri verir; aynı alarm daha sonra test katındaki sol dal bloklu kayıtlarda denenir.

**Durağın sonunda:** Vakanın medyan atımındaki atım sayısını (19), eşiğine ulaşan derivasyon sayısını (9) ve test katında sol dal bloğu olan 54 kayıttan kaçının alarm verdiğini (51) bulacaksınız.

In [ ]:
#@title 2. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
if "ecgs" not in globals():
    ecgs = ptbxl.table()
    signals = ptbxl.signals(ecgs)
    RHYTHMS = ["SR", "AFIB", "STACH", "SBRAD", "SARRH", "PVC"]
    labels = ptbxl.labels(ecgs, RHYTHMS)
    split = ecgs["split"].to_numpy()
    tr, va, te = split == "training", split == "validation", split == "test"

    r_peak_list = []
    for start in range(0, len(signals), 500):
        r_peak_list += beats.r_peaks_many(signals[start:start + 500])
    rr_list = [np.diff(p) / 100 for p in r_peak_list]
    cv = np.array([rr.std(ddof=1) / rr.mean() if len(rr) >= 3 else np.nan for rr in rr_list])


def _course_st():
    """9. ve 10. katlardaki her kaydın medyan atımında 12 derivasyonun J noktasındaki ST sapması (bir kez hesaplanır, yaklaşık yarım dakika)."""
    if "_st_cache" in globals():
        return _st_cache
    rows = []
    for i in np.where(va | te)[0]:
        beat, n_beats = beats.median_beat(signals[i], peaks=r_peak_list[i])
        st = beats.st_deviation(beat)
        rows.append({"ecg_id": int(ecgs["ecg_id"].iloc[i]), "split": split[i], **st["st_mv"]})
    globals()["_st_cache"] = pd.DataFrame(rows)
    return _st_cache

CASE = 13913
_case = int(np.where(ecgs["ecg_id"] == CASE)[0][0])
print("Kayıt", CASE, "·", ecgs["age"].iloc[_case], "yaş,", ecgs["sex"].iloc[_case], "·", ecgs["report"].iloc[_case][:60], "...")
print("2. durak hazır.")

### Adım 2.1: Vakanın medyan atımı

**Amaç.** Bu adımda vakanın tek tek atımlarından, bir EKG cihazının yazılımının ölçümden önce yaptığı gibi, her derivasyon için temiz bir atım oluşturulur.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: ST segmenti ölçülmeden önce 10 saniyelik 12 derivasyonlu bir EKG'nin medyan atımını, yani her derivasyon için temiz bir atımı oluşturmak istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; PTB-XL'in 13913 numaralı kaydını, yani 68 yaşında bir erkeğin 100 Hz'de alınmış 10 saniyelik 12 derivasyonlu EKG'sini tutar. Her satır, bir öncekinden 10 ms sonra alınmış bir örnektir: "zaman" sütunu örneğin zamanını saniye olarak verir, "I" ile "V6" arasındaki on iki sütun on iki derivasyonun değerlerini mV olarak tutar; "R tepesi" sütunu ise R tepelerinin bulunduğu örneklerde 1, diğer örneklerde 0'dır.  
> Görev: Her atımı R tepesinden 0,30 s önceden 0,55 s sonraya kadar kes ve yalnızca penceresi tamamen 10 saniyenin içinde kalan atımları tut. Her derivasyon için bu atımların örnek örnek medyanını al: Medyan atım budur. V2'nin medyan atımını, V2'nin tek tek atımlarının üzerine çiz; tek tek atımlar açık gri olsun ve yatay eksende R tepesine göre zaman ms olarak yer alsın.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Kullanılan atım sayısını yazdır ve "cevap" adıyla sakla.  
> Test: Her atım −300 ile 550 ms arasında 86 örnek içermeli; V2'nin medyan atımı her örnekte tek tek atımların aralığında kalmalı.  

In [ ]:
#@title Adım 2.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_peak = np.zeros(1000, dtype=int)
_peak[r_peak_list[_case]] = 1
veri = pd.concat([pd.DataFrame({"zaman": np.round(np.arange(1000) / 100, 2)}),
                  pd.DataFrame(signals[_case].T, columns=list(beats.LEADS)), pd.DataFrame({"R tepesi": _peak})], axis=1)
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.1 kontrolü
teach.check_step("Adım 2.1", "cevap", globals())

In [ ]:
#@title Adım 2.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_peak = np.zeros(1000, dtype=int)
_peak[r_peak_list[_case]] = 1
veri = pd.concat([pd.DataFrame({"zaman": np.round(np.arange(1000) / 100, 2)}),
                  pd.DataFrame(signals[_case].T, columns=list(beats.LEADS)), pd.DataFrame({"R tepesi": _peak})], axis=1)
tepeler = np.where(veri["R tepesi"].to_numpy() == 1)[0]
derivasyonlar = veri[["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]].to_numpy().T
atimlar = np.stack([derivasyonlar[:, p - 30:p + 56] for p in tepeler if p - 30 >= 0 and p + 56 <= derivasyonlar.shape[1]])
medyan_atim = np.median(atimlar, axis=0)
ms = np.arange(-30, 56) * 10
fig, ax = plt.subplots(figsize=(5, 3.2))
for tek in atimlar[:, 7]:
    ax.plot(ms, tek, color="#BBBBBB", lw=0.7)
ax.plot(ms, medyan_atim[7], color="#9B1B1B", lw=2, label="V2'nin medyan atımı")
ax.axhline(0, color="#999999", lw=0.6)
ax.set_xlabel("R tepesine göre zaman (ms)")
ax.set_ylabel("mV")
ax.legend(fontsize=8)
plt.show()
cevap = len(atimlar)
print("Kullanılan atım:", cevap)
teach.check_step("Adım 2.1", "cevap", globals())

**Sonuç ne gösteriyor?** V2'nin 19 atımı neredeyse üst üste düşer; bu yüzden medyanları temiz bir atımdır. V2'nin ST segmenti QRS kompleksinin hemen ardından taban çizgisinin çok üstüne çıkar ve T dalgasıyla birleşir: Bu, akut bir anterior enfarktüsün biçimidir.

### Adım 2.2: J noktasında ST ve evrensel tanımın ölçütleri

**Amaç.** Bu adımda vakanın on iki derivasyonundaki ST sapması ölçülür ve otomatik bir STEMI alarmının yaptığı gibi evrensel tanımın ölçütleri uygulanır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Bir medyan atımın on iki derivasyonunda J noktasındaki ST sapmasını ölçmek ve EKG'nin miyokard enfarktüsünün dördüncü evrensel tanımındaki ST ölçütlerini karşılayıp karşılamadığını görmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; PTB-XL'in 13913 numaralı kaydının, yani 68 yaşında bir erkeğin EKG'sinin 12 derivasyonuna ait medyan atımı tutar. Medyan atım, R tepelerine göre hizalanmış tek tek atımların örnek örnek medyanıdır. Her satır, bir öncekinden 10 ms sonra gelen bir örnektir: "zaman" sütunu örneğin R tepesine göre zamanını −300 ile 550 ms arasında ms olarak verir, "I" ile "V6" arasındaki on iki sütun ise on iki derivasyonun değerlerini mV olarak tutar. Bu medyan atımda QRS kompleksi, on iki derivasyonun eğimlerinin gösterdiği gibi, R tepesinden 70 ms önce başlar ve R tepesinden 50 ms sonra J noktasında biter.  
> Görev: Her derivasyon için referans düzeyi olarak R tepesinden 130, 120, 110 ve 100 ms önceki değerlerin ortalamasını al; bunlar PR segmentinin, QRS başlangıcından 20 ms önce biten 40 ms'sidir. Bir derivasyonun ST sapması, R tepesinden 50 ms sonraki J noktasındaki değerinden referans düzeyinin çıkarılmasıyla bulunur. Bir derivasyon, J noktasındaki ST yükselmesi kendi eşiğine ulaştığında miyokard enfarktüsünün dördüncü evrensel tanımındaki ölçütü karşılar. Eşikler V2 ve V3 için 40 yaş ve üstündeki erkeklerde 0,2 mV, 40 yaşın altındaki erkeklerde 0,25 mV, kadınlarda 0,15 mV; diğer bütün derivasyonlarda ise 0,1 mV'tur. Aynı gruptaki en az iki derivasyon ölçütü karşıladığında alarm çalar. Ardışık derivasyon grupları anterior (V1-V6), inferior (D2, D3 ve aVF; tablodaki "II", "III" ve "aVF" sütunları) ve lateraldir (D1 ve aVL; tablodaki "I" ve "aVL" sütunları); aVR hiçbir gruba girmez.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her derivasyonun ST sapmasını mV ve mm olarak (1 mm = 0,1 mV), ölçütü karşılayan derivasyonları ve alarmın çaldığı grupları yazdır; ölçütü karşılayan derivasyonların sayısını "cevap" adıyla sakla.  
> Test: D3'ün (tablodaki "III" sütunu) ve aVR'nin ST sapmaları negatif olmalı; çünkü kardiyologların raporu D3'te ST çökmesi tanımlamaktadır.  

In [ ]:
#@title Adım 2.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_beat, _ = beats.median_beat(signals[_case], peaks=r_peak_list[_case])
veri = pd.concat([pd.DataFrame({"zaman": np.arange(-30, 56) * 10}), pd.DataFrame(np.round(_beat.T, 3), columns=list(beats.LEADS))], axis=1)
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.2 kontrolü
teach.check_step("Adım 2.2", "cevap", globals())

In [ ]:
#@title Adım 2.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_beat, _ = beats.median_beat(signals[_case], peaks=r_peak_list[_case])
veri = pd.concat([pd.DataFrame({"zaman": np.arange(-30, 56) * 10}), pd.DataFrame(np.round(_beat.T, 3), columns=list(beats.LEADS))], axis=1)
derivasyonlar = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
t = veri.set_index("zaman")
duzey = t.loc[[-130, -120, -110, -100], derivasyonlar].mean()
st = (t.loc[50, derivasyonlar] - duzey).round(3)
esik = {d: (0.2 if d in ("V2", "V3") else 0.1) for d in derivasyonlar}   # 68 yaşında bir erkek
karsilayan = [d for d in derivasyonlar if st[d] >= esik[d]]
gruplar = {"anterior": ["V1", "V2", "V3", "V4", "V5", "V6"], "inferior": ["II", "III", "aVF"], "lateral": ["I", "aVL"]}
calan = [g for g, adlar in gruplar.items() if sum(d in karsilayan for d in adlar) >= 2]
print(pd.DataFrame({"ST (mV)": st, "ST (mm)": (st * 10).round(1)}).T)
print("Ölçütü karşılayan derivasyonlar:", karsilayan)
print("Alarmın çaldığı gruplar:", calan)
cevap = len(karsilayan)
teach.check_step("Adım 2.2", "cevap", globals())

**Sonuç ne gösteriyor?** 9 derivasyon eşiğine ulaşır ve alarm anterior ile lateral grupta çalar. V3'teki yükselme yaklaşık 10 mm'dir, yani eşiğinin beş katıdır. D3'teki ST çökmesinin D1 ve aVL'deki ST yükselmesiyle birlikte görülmesi, raporun tanımladığı ayna görüntüsü değişikliğidir.

### Adım 2.3: Sol dal bloğunda alarm

**Amaç.** Bu adımda aynı alarm test katındaki bütün kayıtlarda denenir ve ölçütlerin uygulanamadığı sol dal bloğunda ne yaptığı görülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Miyokard enfarktüsünün dördüncü evrensel tanımındaki ST ölçütlerine dayanan otomatik bir STEMI alarmını bir test kümesinin bütün EKG'lerine uygulamak ve alarmın tam sol dal bloğunda ne sıklıkla çaldığını görmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; PTB-XL'in test katındaki 2.198 kaydı tutar. Her satır bir kayıttır: "kayıt" sütunu kaydın numarasını, "cinsiyet" hastanın cinsiyetini ("E" ya da "K"), "yaş" ise yaşını verir; "LBBB" sütunu kardiyologlar tam sol dal bloğu tanımladıysa 1, tanımlamadıysa 0'dır. "I" ile "V6" arasındaki on iki sütun, on iki derivasyonun ST sapmasını mV olarak tutar: Kaydın medyan atımında J noktasındaki değerden PR segmentinin düzeyi çıkarılmıştır.  
> Görev: Bir derivasyon, J noktasındaki ST yükselmesi kendi eşiğine ulaştığında miyokard enfarktüsünün dördüncü evrensel tanımındaki ölçütü karşılar. Eşikler V2 ve V3 için 40 yaş ve üstündeki erkeklerde 0,2 mV, 40 yaşın altındaki erkeklerde 0,25 mV, kadınlarda 0,15 mV; diğer bütün derivasyonlarda ise 0,1 mV'tur. Aynı gruptaki en az iki derivasyon ölçütü karşıladığında alarm çalar. Ardışık derivasyon grupları anterior (V1-V6), inferior (D2, D3 ve aVF; tablodaki "II", "III" ve "aVF" sütunları) ve lateraldir (D1 ve aVL; tablodaki "I" ve "aVL" sütunları); aVR hiçbir gruba girmez. Alarmı her kayda uygula; V2 ve V3'ün eşiklerini kaydın sahibi olan hastanın cinsiyetine ve yaşına göre seç. Bütün kayıtlar arasındaki ve sol dal bloğu olan kayıtlar arasındaki alarmları say.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Bütün kayıtlar arasındaki ve sol dal bloğu olan kayıtlar arasındaki alarm sayılarını kayıt sayılarıyla birlikte yazdır; sol dal bloğu olan kayıtlar arasındaki alarm sayısını "cevap" adıyla sakla.  
> Test: 2.198 kayıt bulunmalı ve bunların 54'ünde sol dal bloğu olmalı; bu kayıtlar arasındaki alarmlar 54'ten fazla olamaz.  

In [ ]:
#@title Adım 2.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_st = _course_st()
_st = _st[_st["split"] == "test"].reset_index(drop=True)
_rows = ecgs.set_index("ecg_id").loc[_st["ecg_id"]]
veri = pd.concat([pd.DataFrame({"kayıt": _st["ecg_id"], "cinsiyet": np.where(_rows["sex"].to_numpy() == "M", "E", "K"),
                                "yaş": _rows["age"].to_numpy().astype(int),
                                "LBBB": _rows["codes"].apply(lambda d: int("CLBBB" in d)).to_numpy()}),
                  _st[list(beats.LEADS)].round(3)], axis=1)
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.3 kontrolü
teach.check_step("Adım 2.3", "cevap", globals())

In [ ]:
#@title Adım 2.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_st = _course_st()
_st = _st[_st["split"] == "test"].reset_index(drop=True)
_rows = ecgs.set_index("ecg_id").loc[_st["ecg_id"]]
veri = pd.concat([pd.DataFrame({"kayıt": _st["ecg_id"], "cinsiyet": np.where(_rows["sex"].to_numpy() == "M", "E", "K"),
                                "yaş": _rows["age"].to_numpy().astype(int),
                                "LBBB": _rows["codes"].apply(lambda d: int("CLBBB" in d)).to_numpy()}),
                  _st[list(beats.LEADS)].round(3)], axis=1)
derivasyonlar = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
gruplar = [["V1", "V2", "V3", "V4", "V5", "V6"], ["II", "III", "aVF"], ["I", "aVL"]]


def alarm_ver(satir):
    v23 = 0.15 if satir["cinsiyet"] == "K" else (0.2 if satir["yaş"] >= 40 else 0.25)
    karsilayan = {d for d in derivasyonlar if satir[d] >= (v23 if d in ("V2", "V3") else 0.1)}
    return any(sum(d in karsilayan for d in g) >= 2 for g in gruplar)


alarm = veri.apply(alarm_ver, axis=1)
lbbb = veri["LBBB"] == 1
print("Bütün kayıtlar:", int(alarm.sum()), "alarm,", len(veri), "kayıt")
print("Sol dal bloğu:", int(alarm[lbbb].sum()), "alarm,", int(lbbb.sum()), "kayıt")
cevap = int(alarm[lbbb].sum())
teach.check_step("Adım 2.3", "cevap", globals())

**Sonuç ne gösteriyor?** Sol dal bloğu olan 54 kaydın 51 tanesi alarm verir. LBBB'de ST segmenti QRS kompleksinin ana sapmasının tersine kayar; bu yüzden V1-V3'te enfarktüs olmadan ST yükselmesi görülür. Evrensel tanım bu nedenle ST ölçütlerini LBBB'de uygulamaz; bir alarmın da önce bloğu tanıması ya da Sgarbossa ölçütleri gibi bu durum için geliştirilmiş ölçütleri kullanması gerekir.

**2. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html#stop-2) dönün ve duraktan sonrasını okumaya devam edin.

## Durak 3: Uzun süreli izlem: Pencereler, AF yükü ve yanlış alarmlar

*Ders: [3. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html#stop-3)* · *Python yolunda aynı sonuç: Kısım 2, 8-9. bölümler*

Sanal kliniğin bir Holter cihazı VC-0128 numaralı hastayı 24 saat izlemiştir. Bu durakta kaydın atımları, bir kayıt cihazının algoritmasının yaptığı gibi 30 saniyelik pencereler hâlinde okunur: Her penceredeki RR aralıklarının düzensizliği alarmı belirler, alarm verilen pencereler AF yükünü verir ve yanlış alarmların hangi saatlerde çıktığı izlenir. Kayıt sentetik olduğu için gerçek durum atım atım bilinmektedir ve hiçbir şeyin indirilmesi gerekmez.

**Durağın sonunda:** Alarm verilen pencere sayısını (2.880 pencereden 519), kestirilen AF yükünü (gerçek yük yüzde 15,4 iken yüzde 18,0) ve yanlış alarmların sayısını (75) hangi saatlerde çıktıklarıyla birlikte bulacaksınız.

In [ ]:
#@title 3. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
if "holter_" not in globals():
    holter_ = holter.recording()
_b = holter_["detected_s"]
_af = np.zeros(len(_b), dtype=int)
for _s, _e in holter_["episodes"]:
    _af[(_b >= _s) & (_b <= _e)] = 1
beats_ = pd.DataFrame({"time": np.round(_b[1:], 3), "RR": np.round(np.diff(_b), 3), "AF": _af[1:]})
print(len(beats_), "atım, 24 saatte")
print("3. durak hazır.")

### Adım 3.1: 30 saniyelik pencerelerde ritmin düzensizliği

**Amaç.** Bu adımda 24 saat, bir monitörün yaptığı gibi pencere pencere okunur ve RR aralıkları düzensiz olan pencereler sayılır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: 24 saatlik bir Holter kaydını 30 saniyelik pencereler hâlinde okumak ve kaç pencerede RR aralıklarının bir atriyal fibrilasyon (AF) alarmı verecek kadar düzensiz olduğunu görmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; sanal kliniğin VC-0128 numaralı hastasına, çarpıntı yakınması olan 76 yaşındaki bir kadına ait 24 saatlik bir Holter kaydının atımlarını, kayıt cihazının atım algılayıcısının bulduğu biçimiyle tutar. Kayıt saat 08.00'de başlar. Her satır bir atımdır: "zaman" sütunu atımın kaydın başından itibaren saniye cinsinden zamanını, "RR" ise önceki atımdan bu yana geçen aralığı saniye olarak verir.  
> Görev: 24 saati, kaydın başından başlayarak 30 saniyelik pencerelere böl ve her atımı zamanının düştüğü pencereye ver. En az beş RR aralığı içeren her pencerede ritmin düzensizliğini değişim katsayısıyla ölç: Penceredeki RR aralıklarının standart sapması (paydada aralık sayısının bir eksiği kullanılır) ortalamalarına bölünür. Her pencerenin değişim katsayısını saate karşı çiz ve 0,10 düzeyinde yatay bir çizgi ekle.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Pencere sayısını yazdır; değişim katsayısı 0,10'un üstünde olan pencerelerin sayısını "cevap" adıyla sakla.  
> Test: 30 saniyelik 2.880 pencere bulunmalı ve bütün değişim katsayıları pozitif olmalı.  

In [ ]:
#@title Adım 3.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = beats_[["time", "RR"]].rename(columns={"time": "zaman"})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.1 kontrolü
teach.check_step("Adım 3.1", "cevap", globals())

In [ ]:
#@title Adım 3.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = beats_[["time", "RR"]].rename(columns={"time": "zaman"})
pencere = (veri["zaman"] // 30).astype(int)
pencereye_gore = veri.groupby(pencere)["RR"]
pencereler = pd.DataFrame({"n": pencereye_gore.count(), "dk": pencereye_gore.std() / pencereye_gore.mean()})
pencereler = pencereler[pencereler["n"] >= 5]
saat = (8 + pencereler.index * 30 / 3600) % 24
fig, ax = plt.subplots(figsize=(9, 3))
ax.scatter(pencereler.index * 30 / 3600, pencereler["dk"], s=3, color="#1F5F8B")
ax.axhline(0.10, color="#9B1B1B", ls="--", lw=1)
ax.set_xticks(range(0, 25, 4))
ax.set_xticklabels([f"{(8 + h) % 24:02d}:00" for h in range(0, 25, 4)])
ax.set_ylim(0, 0.5)
ax.set_xlabel("saat")
ax.set_ylabel("değişim katsayısı")
plt.show()
cevap = int((pencereler["dk"] > 0.10).sum())
print(len(pencereler), "pencere; 0,10'un üstünde:", cevap)
teach.check_step("Adım 3.1", "cevap", globals())

**Sonuç ne gösteriyor?** 2.880 pencerenin 519 tanesinde değişim katsayısı 0,10'un üstündedir. Bunların çoğu öğleden sonra, akşam ve gece görülen üç blokta, yani AF ataklarında yer alır; birkaç dağınık pencere ve çok yüksek değerlerin görüldüğü iki kısa patlama ise bu blokların dışındadır.

### Adım 3.2: AF yükü

**Amaç.** Bu adımda pencereler, bir Holter raporunun verdiği sayıya, yani günün AF içinde geçen bölümünün oranına dönüştürülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: 24 saatlik bir Holter kaydının AF yükünü, yani kaydın atriyal fibrilasyon içinde geçen bölümünün oranını, 30 saniyelik pencereleri okuyan bir algoritmanın bildireceği biçimde öğrenmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; sanal kliniğin VC-0128 numaralı hastasına, çarpıntı yakınması olan 76 yaşındaki bir kadına ait 24 saatlik bir Holter kaydının atımlarını, kayıt cihazının atım algılayıcısının bulduğu biçimiyle tutar. Kayıt saat 08.00'de başlar. Her satır bir atımdır: "zaman" sütunu atımın kaydın başından itibaren saniye cinsinden zamanını, "RR" ise önceki atımdan bu yana geçen aralığı saniye olarak verir.  
> Görev: 24 saati, kaydın başından başlayarak 30 saniyelik pencerelere böl ve her atımı zamanının düştüğü pencereye ver. En az beş RR aralığı içeren her pencerede ritmin düzensizliğini değişim katsayısıyla ölç: Penceredeki RR aralıklarının standart sapması (paydada aralık sayısının bir eksiği kullanılır) ortalamalarına bölünür. Değişim katsayısı 0,10'un üstünde olan pencereye AF de. AF yükü, AF penceresi sayısının 30 saniyeyle çarpımının kaydın 24 saatine bölümüdür. İlk ve son AF penceresinin saatini de bul.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: AF penceresi sayısını yazdır; AF yükünü yüzde olarak bir ondalık basamakla "cevap" adıyla sakla.  
> Test: Yük, AF pencerelerinin 30 saniyelik bütün pencereler içindeki oranının 100 ile çarpımına eşit olmalı.  

In [ ]:
#@title Adım 3.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = beats_[["time", "RR"]].rename(columns={"time": "zaman"})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.2 kontrolü
teach.check_step("Adım 3.2", "cevap", globals())

In [ ]:
#@title Adım 3.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = beats_[["time", "RR"]].rename(columns={"time": "zaman"})
pencere = (veri["zaman"] // 30).astype(int)
pencereye_gore = veri.groupby(pencere)["RR"]
pencereler = pd.DataFrame({"n": pencereye_gore.count(), "dk": pencereye_gore.std() / pencereye_gore.mean()})
pencereler = pencereler[pencereler["n"] >= 5]
af_pencereleri = pencereler.index[pencereler["dk"] > 0.10]
cevap = round(100 * len(af_pencereleri) * 30 / (24 * 3600), 1)
ilk, son = [(8 + w * 30 / 3600) % 24 for w in (af_pencereleri.min(), af_pencereleri.max())]
print(len(af_pencereleri), "AF penceresi; yük:", cevap, "%")
print("ilk ve son AF penceresi:", f"{int(ilk):02d}:{int(ilk % 1 * 60):02d}", f"{int(son):02d}:{int(son % 1 * 60):02d}")
teach.check_step("Adım 3.2", "cevap", globals())

**Sonuç ne gösteriyor?** Pencereler yüzde 18,0 oranında bir AF yükü vermektedir; oysa kardiyoloğun doğruladığı ataklar günün yüzde 15,4'ünü kaplar. Kestirim fazla yüksektir: Atakların dışındaki bazı pencereler de alarm vermiştir. Listedeki ilk ve son AF penceresinin tek bir atağın başı ve sonu olmadığına da dikkat edin; gün içinde üç atak görülmüştür.

### Adım 3.3: Yanlış alarmlar nereden geliyor?

**Amaç.** Bu adımda alarm doğrulanmış ataklarla karşılaştırılır ve bir Holter okuyucusunun yapacağı gibi yanlış alarmların saatleri bulunur.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: 24 saatlik bir Holter kaydının AF alarmını bir kardiyoloğun doğruladığı ataklarla karşılaştırmak ve yanlış alarmların günün hangi saatlerinde çıktığını görmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; sanal kliniğin VC-0128 numaralı hastasına, çarpıntı yakınması olan 76 yaşındaki bir kadına ait 24 saatlik bir Holter kaydının atımlarını, kayıt cihazının atım algılayıcısının bulduğu biçimiyle tutar. Kayıt saat 08.00'de başlar. Her satır bir atımdır: "zaman" sütunu atımın kaydın başından itibaren saniye cinsinden zamanını, "RR" ise önceki atımdan bu yana geçen aralığı saniye olarak verir. "AF" sütunu, bir kardiyoloğun kaydın tamamında doğruladığı atriyal fibrilasyon ataklarındaki atımlar için 1, öteki bütün atımlar için 0'dır.  
> Görev: 24 saati, kaydın başından başlayarak 30 saniyelik pencerelere böl ve her atımı zamanının düştüğü pencereye ver. En az beş RR aralığı içeren her pencerede ritmin düzensizliğini değişim katsayısıyla ölç: Penceredeki RR aralıklarının standart sapması (paydada aralık sayısının bir eksiği kullanılır) ortalamalarına bölünür. Değişim katsayısı 0,10'un üstünde olan pencereye AF de. Atımlarının yarısından fazlası doğrulanmış bir atağa ait olan pencere gerçek bir AF penceresidir. Yanlış alarmları, yani gerçek AF penceresi olmayan AF pencerelerini, ve kaçırılan pencereleri, yani alarm verilmeyen gerçek AF pencerelerini say. Ardından yanlış alarmları her saat için ayrı ayrı say.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Kaçırılan pencere sayısını ve yanlış alarmların saatlere göre dağılımını gösteren küçük bir tablo yazdır; yanlış alarmların toplam sayısını "cevap" adıyla sakla.  
> Test: Bütün saatlerdeki yanlış alarmların toplamı genel toplama eşit olmalı; AF pencereleri de alarm verilen gerçek AF pencereleriyle yanlış alarmların toplamına eşit olmalı.  

In [ ]:
#@title Adım 3.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = beats_.rename(columns={"time": "zaman"})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.3 kontrolü
teach.check_step("Adım 3.3", "cevap", globals())

In [ ]:
#@title Adım 3.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = beats_.rename(columns={"time": "zaman"})
pencere = (veri["zaman"] // 30).astype(int)
pencereye_gore = veri.groupby(pencere)
pencereler = pd.DataFrame({"n": pencereye_gore["RR"].count(),
                        "dk": pencereye_gore["RR"].std() / pencereye_gore["RR"].mean(),
                        "gercek_af": pencereye_gore["AF"].mean() > 0.5})
pencereler = pencereler[pencereler["n"] >= 5]
alarm = pencereler["dk"] > 0.10
yanlis_alarm = alarm & ~pencereler["gercek_af"]
kacirilan = int((~alarm & pencereler["gercek_af"]).sum())
saat = ((8 + pencereler.index * 30 / 3600) % 24).astype(int)
saate_gore = pd.Series(yanlis_alarm.to_numpy(), index=pd.Index(saat, name="saat")).groupby(level=0).sum()
print("kaçırılan pencere:", kacirilan)
print(saate_gore[saate_gore > 0].rename("yanlış alarm"))
cevap = int(yanlis_alarm.sum())
print("toplam yanlış alarm:", cevap)
teach.check_step("Adım 3.3", "cevap", globals())

**Sonuç ne gösteriyor?** Alarm hiçbir AF penceresini kaçırmamış, ancak neredeyse tamamı üç saatte toplanan 75 yanlış alarm vermiştir. Saat 07.00'de ve 23.00'te elektrotlar gürültülüydü; cihazın algılayıcısı bazı atımları kaçırmış, bazılarını ise olmadığı hâlde bulmuştur. Bu yüzden aralıklar AF'dekinden çok daha fazla dağılır. 18.00 ile 19.00 arasında ise hastada her biri kısa ve uzun bir aralık oluşturan çok sayıda atriyal erken vuru vardı. Kaydı okuyan bir hekim bu üç durumu kayıtta bir bakışta görür; bir algoritmanın ise bir kalite kuralına ve erken vurular için bir kurala ihtiyacı vardır. Ders sayfasındaki animasyon ve Python yolu bu kuralları dener.

**3. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html#stop-3) dönün ve duraktan sonrasını okumaya devam edin.

## Prompt yolunda haftalık proje

*Haftanın projesi: [5. haftanın README sayfası, Haftalık proje](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-05/README.md#haftal%C4%B1k-proje)*

Haftalık proje, her adım için bir prompt olmak üzere aşağıdaki promptlarla da yapılabilir. Önce hazırlık hücresini bir kez çalıştırın; hücre projenin verilerini yükler. Ardından her adım, duraklardaki adımlar gibi ilerler. Tek fark şudur: Projede kontrol hücresi ve hazır kod yoktur, çünkü sonuçlar rapora girer. Bir promptun testi sağlanmadan sonraki prompta geçilmez ve her prompt YZ kullanım günlüğüne yazılır.

In [ ]:
#@title Projenin hazırlığı: Bu hücreyi bir kez çalıştırın (projenin verilerini yükler)
if "ecgs" not in globals():
    ecgs = ptbxl.table()
    signals = ptbxl.signals(ecgs)
    RHYTHMS = ["SR", "AFIB", "STACH", "SBRAD", "SARRH", "PVC"]
    labels = ptbxl.labels(ecgs, RHYTHMS)
    split = ecgs["split"].to_numpy()
    tr, va, te = split == "training", split == "validation", split == "test"

    r_peak_list = []
    for start in range(0, len(signals), 500):
        r_peak_list += beats.r_peaks_many(signals[start:start + 500])
    rr_list = [np.diff(p) / 100 for p in r_peak_list]
    cv = np.array([rr.std(ddof=1) / rr.mean() if len(rr) >= 3 else np.nan for rr in rr_list])


def _course_st():
    """9. ve 10. katlardaki her kaydın medyan atımında 12 derivasyonun J noktasındaki ST sapması (bir kez hesaplanır, yaklaşık yarım dakika)."""
    if "_st_cache" in globals():
        return _st_cache
    rows = []
    for i in np.where(va | te)[0]:
        beat, n_beats = beats.median_beat(signals[i], peaks=r_peak_list[i])
        st = beats.st_deviation(beat)
        rows.append({"ecg_id": int(ecgs["ecg_id"].iloc[i]), "split": split[i], **st["st_mv"]})
    globals()["_st_cache"] = pd.DataFrame(rows)
    return _st_cache

_codes = ecgs.set_index("ecg_id")["codes"]


def _project_table():
    _st = _course_st()
    _rows = ecgs.set_index("ecg_id").loc[_st["ecg_id"]]
    return pd.concat([pd.DataFrame({"kayıt": _st["ecg_id"], "küme": ["doğrulama" if s == "validation" else "test" for s in _st["split"]],
                                    "cinsiyet": np.where(_rows["sex"].to_numpy() == "M", "E", "K"),
                                    "yaş": _rows["age"].to_numpy().astype(int),
                                    "akut enfarktüs": _rows["acute_mi"].to_numpy().astype(int),
                                    "ifadeler": [", ".join(sorted(d)) for d in _rows["codes"]]}),
                      _st[list(beats.LEADS)].round(3)], axis=1)
print("Proje hazır.")

### Proje adımı 1: Alarm ve akut enfarktüsler

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Miyokard enfarktüsünün dördüncü evrensel tanımındaki ST ölçütlerine dayanan otomatik bir STEMI alarmını bir göğüs ağrısı biriminin yapacağı gibi değerlendirmek istiyorum: Alarm kaç akut enfarktüsü bulur ve alarmlarının kaçı doğrudur?  
> Veri: Defterde "veri" adında bir tablo var; PTB-XL'in 9. ve 10. katlarındaki 4.381 kaydı tutar. Her satır bir kayıttır: "kayıt" sütunu kaydın numarasını, "küme" kümesini ("doğrulama" ya da "test"), "cinsiyet" hastanın cinsiyetini ("E" ya da "K"), "yaş" ise yaşını verir. "akut enfarktüs" sütunu, kardiyologlar kayıttaki bir enfarktüsü akut (I. ya da I-II. evre) olarak sınıfladıysa 1, sınıflamadıysa 0'dır. "ifadeler" sütunu kardiyologların bütün ifadelerinin kodlarını virgülle ayrılmış olarak listeler; örneğin tam sol dal bloğu için "CLBBB", sol ventrikül hipertrofisi için "LVH", kalp pili için "PACE". "I" ile "V6" arasındaki on iki sütun, on iki derivasyonun ST sapmasını mV olarak tutar: Kaydın medyan atımında J noktasındaki değerden PR segmentinin düzeyi çıkarılmıştır.  
> Görev: Bir derivasyon, J noktasındaki ST yükselmesi kendi eşiğine ulaştığında miyokard enfarktüsünün dördüncü evrensel tanımındaki ölçütü karşılar. Eşikler V2 ve V3 için 40 yaş ve üstündeki erkeklerde 0,2 mV, 40 yaşın altındaki erkeklerde 0,25 mV, kadınlarda 0,15 mV; diğer bütün derivasyonlarda ise 0,1 mV'tur. Aynı gruptaki en az iki derivasyon ölçütü karşıladığında alarm çalar. Ardışık derivasyon grupları anterior (V1-V6), inferior (D2, D3 ve aVF; tablodaki "II", "III" ve "aVF" sütunları) ve lateraldir (D1 ve aVL; tablodaki "I" ve "aVL" sütunları); aVR hiçbir gruba girmez. Alarmı her kayda uygula; V2 ve V3'ün eşiklerini hastanın cinsiyetine ve yaşına göre seç. Alarmı "akut enfarktüs" sütunuyla karşılaştır; duyarlılığını, özgüllüğünü ve pozitif öngörü değerini bütün kayıtlar için ve her küme için ayrı ayrı hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Kayıt, akut enfarktüs ve alarm sayılarını ve üç değeri gösteren bir tablo göster; tabloda her küme için bir satır ve bütün kayıtlar için bir satır olsun. Bütün kayıtlardaki duyarlılığı üç ondalık basamakla "cevap" adıyla sakla.  
> Test: 4.381 kayıt ve 26 akut enfarktüs bulunmalı; doğru alarmlar 26'dan fazla olamaz.  

In [ ]:
#@title Proje adımı 1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _project_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 2: Yanlış alarmlar nereden geliyor?

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Otomatik bir STEMI alarmının yanlış alarmlarının hangi tanıları taşıdığını öğrenmek ve böylece ölçütlerinin nerede uymadığını görmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; PTB-XL'in 9. ve 10. katlarındaki 4.381 kaydı tutar. Her satır bir kayıttır: "kayıt" sütunu kaydın numarasını, "küme" kümesini ("doğrulama" ya da "test"), "cinsiyet" hastanın cinsiyetini ("E" ya da "K"), "yaş" ise yaşını verir. "akut enfarktüs" sütunu, kardiyologlar kayıttaki bir enfarktüsü akut (I. ya da I-II. evre) olarak sınıfladıysa 1, sınıflamadıysa 0'dır. "ifadeler" sütunu kardiyologların bütün ifadelerinin kodlarını virgülle ayrılmış olarak listeler; örneğin tam sol dal bloğu için "CLBBB", sol ventrikül hipertrofisi için "LVH", kalp pili için "PACE". "I" ile "V6" arasındaki on iki sütun, on iki derivasyonun ST sapmasını mV olarak tutar: Kaydın medyan atımında J noktasındaki değerden PR segmentinin düzeyi çıkarılmıştır.  
> Görev: Bir derivasyon, J noktasındaki ST yükselmesi kendi eşiğine ulaştığında miyokard enfarktüsünün dördüncü evrensel tanımındaki ölçütü karşılar. Eşikler V2 ve V3 için 40 yaş ve üstündeki erkeklerde 0,2 mV, 40 yaşın altındaki erkeklerde 0,25 mV, kadınlarda 0,15 mV; diğer bütün derivasyonlarda ise 0,1 mV'tur. Aynı gruptaki en az iki derivasyon ölçütü karşıladığında alarm çalar. Ardışık derivasyon grupları anterior (V1-V6), inferior (D2, D3 ve aVF; tablodaki "II", "III" ve "aVF" sütunları) ve lateraldir (D1 ve aVL; tablodaki "I" ve "aVL" sütunları); aVR hiçbir gruba girmez. Alarmı her kayda uygula; V2 ve V3'ün eşiklerini hastanın cinsiyetine ve yaşına göre seç. Akut enfarktüsü olmayan bir kayıttaki alarm yanlış alarmdır. Her ifade için o ifadeyi taşıyan kayıtlardaki yanlış alarmları say; "CLBBB", "PACE" ve "LVH" ifadeleri için ayrıca bu ifadeleri taşıyan kayıtların ne kadarının alarm verdiğini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: En çok yanlış alarm veren 15 ifadeyi göster; "CLBBB" taşıyan kayıtlardaki yanlış alarm sayısını "cevap" adıyla sakla.  
> Test: Bir ifadenin yanlış alarmları, o ifadeyi taşıyan kayıtlardan fazla olamaz; yanlış alarmların toplamı da 1. adımdaki alarmlardan doğru alarmların çıkarılmasıyla bulunan sayıya eşit olmalı.  

In [ ]:
#@title Proje adımı 2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _project_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 3: Eşiklerin taranması

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Otomatik bir STEMI alarmının bütün eşikleri birlikte yükseltildiğinde duyarlılığının ve pozitif öngörü değerinin nasıl değiştiğini görmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; PTB-XL'in 9. ve 10. katlarındaki 4.381 kaydı tutar. Her satır bir kayıttır: "kayıt" sütunu kaydın numarasını, "küme" kümesini ("doğrulama" ya da "test"), "cinsiyet" hastanın cinsiyetini ("E" ya da "K"), "yaş" ise yaşını verir. "akut enfarktüs" sütunu, kardiyologlar kayıttaki bir enfarktüsü akut (I. ya da I-II. evre) olarak sınıfladıysa 1, sınıflamadıysa 0'dır. "ifadeler" sütunu kardiyologların bütün ifadelerinin kodlarını virgülle ayrılmış olarak listeler; örneğin tam sol dal bloğu için "CLBBB", sol ventrikül hipertrofisi için "LVH", kalp pili için "PACE". "I" ile "V6" arasındaki on iki sütun, on iki derivasyonun ST sapmasını mV olarak tutar: Kaydın medyan atımında J noktasındaki değerden PR segmentinin düzeyi çıkarılmıştır.  
> Görev: Bir derivasyon, J noktasındaki ST yükselmesi kendi eşiğine ulaştığında miyokard enfarktüsünün dördüncü evrensel tanımındaki ölçütü karşılar. Eşikler V2 ve V3 için 40 yaş ve üstündeki erkeklerde 0,2 mV, 40 yaşın altındaki erkeklerde 0,25 mV, kadınlarda 0,15 mV; diğer bütün derivasyonlarda ise 0,1 mV'tur. Aynı gruptaki en az iki derivasyon ölçütü karşıladığında alarm çalar. Ardışık derivasyon grupları anterior (V1-V6), inferior (D2, D3 ve aVF; tablodaki "II", "III" ve "aVF" sütunları) ve lateraldir (D1 ve aVL; tablodaki "I" ve "aVL" sütunları); aVR hiçbir gruba girmez. Alarmı her kayda uygula; V2 ve V3'ün eşiklerini hastanın cinsiyetine ve yaşına göre seç. Alarmı her eşiğe sırasıyla 0; 0,05; 0,10; 0,15 ve 0,20 mV eklenmiş olarak tekrarla. Her pay için "akut enfarktüs" sütununa göre duyarlılığı, özgüllüğü ve pozitif öngörü değerini hesapla; duyarlılığı ve pozitif öngörü değerini paya karşı çiz.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her pay için bir satır içeren bir tablo göster; 0,10 mV paydaki duyarlılığı üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Pay büyüdükçe alarm sayısı artamaz; payın 0 olduğu satır 1. adımın sonucuna eşit olmalı.  

In [ ]:
#@title Proje adımı 3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _project_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 4: Ölçütlerin uygulanamadığı kayıtları dışarıda bırakmak

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Otomatik bir STEMI alarmını yalnızca evrensel tanımın ST ölçütlerinin uygulanabildiği EKG'lerde değerlendirmek; sol dal bloğunu ve kalp pili ritmini dışarıda bırakmak istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; PTB-XL'in 9. ve 10. katlarındaki 4.381 kaydı tutar. Her satır bir kayıttır: "kayıt" sütunu kaydın numarasını, "küme" kümesini ("doğrulama" ya da "test"), "cinsiyet" hastanın cinsiyetini ("E" ya da "K"), "yaş" ise yaşını verir. "akut enfarktüs" sütunu, kardiyologlar kayıttaki bir enfarktüsü akut (I. ya da I-II. evre) olarak sınıfladıysa 1, sınıflamadıysa 0'dır. "ifadeler" sütunu kardiyologların bütün ifadelerinin kodlarını virgülle ayrılmış olarak listeler; örneğin tam sol dal bloğu için "CLBBB", sol ventrikül hipertrofisi için "LVH", kalp pili için "PACE". "I" ile "V6" arasındaki on iki sütun, on iki derivasyonun ST sapmasını mV olarak tutar: Kaydın medyan atımında J noktasındaki değerden PR segmentinin düzeyi çıkarılmıştır.  
> Görev: İfadeleri arasında "CLBBB" ya da "PACE" bulunan her kaydı dışarıda bırak. Bir derivasyon, J noktasındaki ST yükselmesi kendi eşiğine ulaştığında miyokard enfarktüsünün dördüncü evrensel tanımındaki ölçütü karşılar. Eşikler V2 ve V3 için 40 yaş ve üstündeki erkeklerde 0,2 mV, 40 yaşın altındaki erkeklerde 0,25 mV, kadınlarda 0,15 mV; diğer bütün derivasyonlarda ise 0,1 mV'tur. Aynı gruptaki en az iki derivasyon ölçütü karşıladığında alarm çalar. Ardışık derivasyon grupları anterior (V1-V6), inferior (D2, D3 ve aVF; tablodaki "II", "III" ve "aVF" sütunları) ve lateraldir (D1 ve aVL; tablodaki "I" ve "aVL" sütunları); aVR hiçbir gruba girmez. Alarmı her kayda uygula; V2 ve V3'ün eşiklerini hastanın cinsiyetine ve yaşına göre seç. Kalan kayıtlarda "akut enfarktüs" sütununa göre duyarlılığı, özgüllüğü ve pozitif öngörü değerini hesapla ve bunları 1. adımla karşılaştır.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Dışarıda bırakılan kayıt sayısını ve üç değeri yazdır; pozitif öngörü değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Dışarıda bırakılan ve kalan kayıtların toplamı 4.381 olmalı.  

In [ ]:
#@title Proje adımı 4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _project_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

<a id="part-2"></a>

# Kısım 2: Python yolu

**Kimler için?** Bu kısım Python yazmayı öğrenmek isteyen katılımcılar içindir. Kısım 1'i çalışmış olmanız gerekmez: 0. bölümden sonra doğrudan buradan başlayın.

**Nasıl ilerler?** Hücreleri yukarıdan aşağıya Shift+Enter ile çalıştırın. Baştaki *Python adımı*, haftanın gerektirdiği dil bilgisini öğretir. Numaralı bölümlerdeki alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerde durur ve ancak denedikten sonra açılmalıdır. *Gözlemle ve düşün* hücreleri açık uçlu sorular sorar; bu soruların çözüm hücresi yoktur. *Python notu* hücreleri, yeni bir yapının ilk kullanıldığı yerde dili açıklar. *İleri düzey (isteğe bağlı)* olarak işaretlenen hücreler, programlama bilenler için ek çalışmalardır. Bu kısma ortadan başlayan katılımcı, o bölümün başındaki *Telafi* hücresini çalıştırır. Bazı alıştırma hücreleri `veri` adlı bir tabloyu hazırlayan bir satırla başlar. Bu tablo, etkileşimli labın prompt atölyesindeki alıştırma promptları için hazırlanmıştır; Python yolunda alıştırma, metninde adı geçen tablolarla çözülebilir.

## Python adımı 5: Batch'ler hâlinde sinyaller

Bu haftanın verileri sinyallerdir: Sıralarıyla saklanan sayılardan oluşan on iki derivasyon ve bunlardan oluşan binlerce kayıt. Bu adımda sanal kliniğin sentetik EKG'leriyle çalışılır. Bir EKG derivasyon × örnek biçiminde bir diziye dönüştürülür, pencerelere bölünür; 40 hastanın EKG'si, her hasta için birkaç etiketle birlikte tek bir dizide toplanır ve bir ağın yaptığı gibi batch'ler hâlinde dolaşılır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
cohort = vclinic.make_cohort(500)
ecg = vclinic.patient_ecg(cohort, "VC-0128", fs=100)
x = ecg.leads.to_numpy().T
print(type(x).__name__, x.shape, x.dtype)
print("D2 derivasyonu, ilk beş örnek (mV):", x[1, :5].round(3))
print("Bütün derivasyonların ilk saniyesi:", x[:, :100].shape)

**Python notu.** `ecg.leads`, her derivasyon için bir sütun ve her örnek için bir satır içeren bir tablodur. `.to_numpy()` bu tabloyu bir sayı dizisine dönüştürür; `.T` ise diziyi *devrik* hâle getirir, yani satırlarla sütunların yerini değiştirir. Artık ilk indis derivasyonu, ikinci indis örneği seçer; bu, PTB-XL'in ve haftanın ağlarının kullandığı sıradır. Python saymaya 0'dan başlar; bu yüzden `x[1]` D2 derivasyonudur. `x[:, :100]` bütün derivasyonları (`:`) ve baştan 100. örneğe kadar olan, 100. örneğin kendisini içermeyen örnekleri, yani 100 Hz'de ilk saniyeyi seçer.

In [ ]:
# Python alıştırması: bir kaydı pencerelere bölmek
fs = 100
length = 2 * fs                       # 2 saniyelik pencereler
starts = list(range(0, x.shape[1] - length + 1, length))
windows = np.stack([x[:, s:s + length] for s in starts])
print(starts)
print(windows.shape)                  # (pencere, derivasyon, örnek)

**Python notu.** `range(başlangıç, bitiş, adım)`, `başlangıç` değerinden `adım` büyüklüğünde adımlarla sayar ve `bitiş` değerine gelmeden durur; burada her pencerenin ilk örneğini verir. `[x[:, s:s + length] for s in starts]` bir *liste üreteci* (list comprehension) örneğidir: `s`'nin her değeri için bir pencere içeren bir liste kurar. `np.stack` pencereleri yeni bir ilk eksen boyunca üst üste koyar; böylece 12 derivasyon × 200 örnekten oluşan beş pencere (5, 12, 200) boyutunda tek bir diziye dönüşür.

In [ ]:
# Python alıştırması: birçok kayıt tek bir dizide, her biri birkaç etiketle
first = cohort.iloc[:40]
X = np.stack([vclinic.patient_ecg(cohort, p, fs=100).leads.to_numpy().T for p in first["patient_id"]])
labels_vc = pd.DataFrame({"AF": (first["ecg_rhythm"] == "AF").astype(int),
                          "LBBB": (first["ecg_conduction"] == "LBBB").astype(int),
                          "STEMI": (first["stemi_territory"] != "none").astype(int)})
labels_vc.index = first["patient_id"]
print(X.shape)
print(labels_vc.sum())
print(labels_vc[labels_vc.sum(axis=1) > 0])

**Python notu.** `(first["ecg_rhythm"] == "AF")` her hastayı "AF" ile karşılaştırır ve `True` ya da `False` verir; `.astype(int)` bunları 1 ve 0'a çevirir. Her etiket için 0 ve 1 değerlerinden oluşan bir sütunu bulunan tabloya *multi-hot* tablo denir: Bir hastanın aynı anda birkaç sütunda 1 değeri olabilir. `labels_vc.sum()` her sütunu, yani her etiketi taşıyan hasta sayısını toplar; `labels_vc.sum(axis=1)` ise her satırı, yani her hastanın etiket sayısını toplar.

In [ ]:
# Python alıştırması: kayıtları batch'ler hâlinde dolaşmak
batch = 16
for start in range(0, len(X), batch):
    part = X[start:start + batch]
    print(start, part.shape, "D2'nin en büyük değeri:", round(float(part[:, 1].max()), 2))

**Python notu.** Bir ağ kayıtlarını *batch'ler* hâlinde, her seferinde birkaçını okuyarak işler; çünkü binlerce kayıt hesaplamanın tek bir adımına sığmaz. `X[start:start + batch]` 16 kayıt alır; son batch kalan 8 kaydı içerir, çünkü bir dilim son elemanın ötesinde bitebilir. `part[:, 1]` batch'teki her kaydın D2 derivasyonudur.

**Hızlı kontrol.** EKG'lerden oluşan bir dizinin boyutu (40, 12, 1000)'dir. `X[5, 1]` neyi tutar?

<details><summary>Cevap</summary>

Altıncı kaydın D2 derivasyonunu, yani 1.000 sayıyı tutar: İlk indis kaydı, ikinci indis derivasyonu seçer; ikisi de 0'dan sayılır.

</details>

**Alıştırma 5.1.** `x` kaydı, örtüşmeyen 2,5 saniyelik (250 örneklik) kaç pencereye bölünür? Pencereleri `starts` örneğindeki gibi kurun.

In [ ]:
veri = alistirma_tablosu("5.1")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma 5.1", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = len(range(0, x.shape[1] - 250 + 1, 250))
teach.check("Alıştırma 5.1", cevap)

**Alıştırma 5.2.** `labels_vc` tablosundaki 40 hastadan kaçı üç etiketten en az ikisini taşır?

In [ ]:
veri = alistirma_tablosu("5.2")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma 5.2", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int((labels_vc.sum(axis=1) >= 2).sum())
teach.check("Alıştırma 5.2", cevap)

**Alıştırma 5.3.** `X` dizisini 16 kayıtlık batch'ler hâlinde dolaşın ve 40 kaydın her birinin D2 derivasyonundaki en büyük değeri toplayın. Bu 40 değerin ortalaması kaç mV'tur? Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
veri = alistirma_tablosu("5.3")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma 5.3", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
largest = []
for start in range(0, len(X), 16):
    largest += list(X[start:start + 16, 1].max(axis=1))
cevap = round(float(np.mean(largest)), 3)
teach.check("Alıştırma 5.3", cevap)

## 1. PTB-XL'in 12 derivasyonlu EKG'leri

*Ders: [Zaman serisi olarak EKG](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html#the-ecg-as-a-time-series)*

PTB-XL, hastaya göre on kata bölünmüş 10 saniyelik 21.799 klinik 12 derivasyonlu EKG içerir. Eğitimde bunlardan dördü kullanılır: 1. ve 2. katlar eğitim, 9. kat doğrulama, 10. kat ise test içindir. `ptbxl.table()` her kayıt için bir satır verir; satırda hasta, yaş ve cinsiyet, kat ve kardiyologların ifadeleri bulunur.

**Formüllerden koda.** Derivasyon başına $N = f_s \times T = 100 \times 10 = 1000$ örnek vardır: `signals.shape` (8737, 12, 1000)'dir. `ptbxl.signals(ecgs)` bütün kayıtları 500'erli batch'ler hâlinde tek bir diziye okur ve diske bir kopyasını yazar; böylece ikinci çalıştırma bir saniye sürer. `ptbxl.labels(ecgs, RHYTHMS)` eğitimin altı etiketi için multi-hot tabloyu kurar.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 1. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB05_zaman_serisi_ve_ekg_yapay_zekasi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(1)

In [ ]:
ecgs = ptbxl.table()
signals = ptbxl.signals(ecgs)
RHYTHMS = ["SR", "AFIB", "STACH", "SBRAD", "SARRH", "PVC"]
labels = ptbxl.labels(ecgs, RHYTHMS)
split = ecgs["split"].to_numpy()
tr, va, te = split == "training", split == "validation", split == "test"

print(signals.shape, signals.dtype)
print(ecgs.groupby("split").agg(kayit=("ecg_id", "size"), hasta=("patient_id", "nunique")))

**Python notu.** `ecgs` her kayıt için bir satır tutar. `codes` sütunu bir kaydın ifadelerini bir *sözlük*, yani kod ve sayı çiftleri olarak saklar; örneğin `{"NORM": 100.0, "SR": 0.0}`. Sayı, kardiyologların bir tanı ifadesine verdiği olasılıktır; ritim ve biçim ifadelerinde 0'dır. `ecgs.groupby("split").agg(...)` satırları kümeye göre gruplar ve aynı anda iki sütun hesaplar: kayıt sayısı ve farklı hasta sayısı (`nunique`).

In [ ]:
print(labels.groupby(split).sum().T)
patients_of = {s: set(ecgs.loc[split == s, "patient_id"]) for s in ("training", "validation", "test")}
print("İki kümede birden bulunan hasta:", len(patients_of["training"] & patients_of["test"]) + len(patients_of["training"] & patients_of["validation"]) + len(patients_of["validation"] & patients_of["test"]))

**Python notu.** `labels.groupby(split).sum()` her kümede her etiketi taşıyan kayıtları sayar; `.T` tabloyu çevirerek etiketleri satırlara taşır. `set(...)` her hastayı bir kez tutar; `&` ise iki kümenin ortak hastalarını verir. 0 sonucu PTB-XL'in katlarına ait kuralı gösterir: Bir hastanın bütün kayıtları aynı kattadır.

In [ ]:
# Haftanın vakası: Test katındaki 13913 numaralı kayıt
CASE = 13913
case = int(np.where(ecgs["ecg_id"] == CASE)[0][0])
print(ecgs.loc[case, ["ecg_id", "age", "sex", "split", "codes"]].to_dict())
print(ecgs.loc[case, "report"])
case_leads = pd.DataFrame(signals[case].T, columns=list(beats.LEADS))
teach.plot_12lead(np.arange(1000) / 100, case_leads, title="PTB-XL, 13913 numaralı kayıt (CC BY 4.0)")
plt.show()

**Gözlemle ve düşün.** Vakanın 12 derivasyonunu bir program okumadan önce siz okuyun. Hangi ritmi görüyorsunuz ve ST segmenti hangi derivasyonlarda yükselmiş ya da çökmüş? Cevabınızı saklayın: 4. ve 6. bölümler onu ağla ve eğitimin alarmıyla karşılaştırır.

**Alıştırma A.** Test katındaki kayıtlardan kaçı hem sinüs ritmi (SR) hem de ventriküler erken vurular (PVC) taşır?

In [ ]:
veri = alistirma_tablosu("A")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma A", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int(((labels["SR"].to_numpy() == 1) & (labels["PVC"].to_numpy() == 1) & te).sum())
teach.check("Alıştırma A", cevap)

## 2. RR aralıkları ve AF için bir kural

*Ders: [Önce bir kural: RR aralıkları ve atriyal fibrilasyon](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html#a-rule-first-rr-intervals-and-atrial-fibrillation)*

Dersteki kural, RR aralıklarının değişim katsayısı 0,10'un üstünde olan kayda AF der. R tepeleri 1. haftanın yöntemiyle, yani Pan-Tompkins sinyaliyle bulunur; yöntem on iki derivasyonun hepsine birden uygulanır. QRS kompleksi küçük olan bir derivasyon, kompleksi büyük olan bir derivasyon kadar ağırlık taşır; böylece düz bir derivasyon bir atımı gizleyemez.

**Formüllerden koda.** R tepeleri örnek cinsinden $r_i$ olmak üzere $RR_i = (r_{i+1} - r_i)/f_s$: `np.diff(p) / 100`. $s_{RR}$'nin paydasında $m - 1$ kullanılarak $\mathrm{DK} = s_{RR}/\overline{RR}$: `rr.std(ddof=1) / rr.mean()`. `beats.r_peaks_many(signals[start:start + 500])` 500 kaydın R tepelerini aynı anda bulur.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 2. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB05_zaman_serisi_ve_ekg_yapay_zekasi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(2)

In [ ]:
r_peak_list = []
for start in range(0, len(signals), 500):
    r_peak_list += beats.r_peaks_many(signals[start:start + 500])
rr_list = [np.diff(p) / 100 for p in r_peak_list]
cv = np.array([rr.std(ddof=1) / rr.mean() if len(rr) >= 3 else np.nan for rr in rr_list])

print(len(r_peak_list), "kayıt;", "kayıt başına RR aralığı (medyan):", int(np.median([len(r) for r in rr_list])))

**Python notu.** `r_peak_list += ...` her batch'in R tepelerini listenin sonuna ekler. `np.diff(p)` komşu R tepeleri arasındaki farkları örnek cinsinden verir; 100'e bölmek bunları saniyeye çevirir. `ddof=1`, dersteki formülde olduğu gibi standart sapmanın paydasına $m$ yerine $m - 1$ koyar. `if len(rr) >= 3 else np.nan`, üçten az aralığı olan bir kayda *sayı değil* (not a number) yazar; böylece kayıt düzenli diye geçemez.

In [ ]:
# Test katından dört kaydın RR aralıkları
fig, axes = plt.subplots(1, 4, figsize=(12, 2.6), sharey=True)
for ax, record in zip(axes, (9, 318, 76, 452)):
    i = int(np.where(ecgs["ecg_id"] == record)[0][0])
    labels_i = [c for c in RHYTHMS if labels.iloc[i][c] == 1]
    ax.plot(rr_list[i] * 1000, marker="o", ms=3)
    ax.set_title(f"{record}: {', '.join(labels_i)} · DK {cv[i]:.3f}", fontsize=9)
    ax.set_xlabel("aralık")
axes[0].set_ylabel("RR (ms)")
plt.show()

In [ ]:
def macro_auc(y_true, prob):
    """Etiketlerin AUC değerlerinin ortalaması: Her sütun için bir AUC."""
    return float(np.mean([roc_auc_score(y_true[:, k], prob[:, k]) for k in range(y_true.shape[1])]))


def youden_threshold(y_true, prob):
    """Duyarlılık + özgüllük toplamının en yüksek olduğu eşik."""
    fpr, tpr, thresholds = roc_curve(y_true, prob)
    return float(thresholds[np.argmax(tpr - fpr)])


def se_sp_ppv(y_true, alarm):
    """Evet ya da hayır biçimindeki bir kararın duyarlılığı, özgüllüğü ve pozitif öngörü değeri."""
    y_true, alarm = np.asarray(y_true, dtype=bool), np.asarray(alarm, dtype=bool)
    tp, fp = int((alarm & y_true).sum()), int((alarm & ~y_true).sum())
    return tp / y_true.sum(), 1 - fp / (~y_true).sum(), tp / max(1, tp + fp)

af = labels["AFIB"].to_numpy() == 1
has_cv = np.isfinite(cv)
for name, m in (("doğrulama", va), ("test", te)):
    m = m & has_cv
    se, sp, ppv = se_sp_ppv(af[m], cv[m] > 0.10)
    print(f"{name}: AUC {roc_auc_score(af[m], cv[m]):.3f} · duyarlılık {se:.3f} · özgüllük {sp:.3f} · PPV {ppv:.3f}")
false_alarm = te & has_cv & (cv > 0.10) & ~af
print("Test katındaki yanlış alarmlar:", int(false_alarm.sum()), labels[false_alarm].sum().to_dict())

**Python notu.** `se_sp_ppv` evet ya da hayır biçimindeki bir kararın doğru ve yanlış alarmlarını sayar ve üç sayı döndürür; `se, sp, ppv = ...` bunları açar. `roc_auc_score(af[m], cv[m])` kararı değil değişim katsayısının kendisini kullanır ve bütün eşikler üzerinden AUC'yi verir. `labels[false_alarm].sum()` yanlış alarm verilen kayıtların etiketlerini sayar; bir kayıt bunlardan birkaçına birden katkıda bulunabilir.

**Gözlemle ve düşün.** Kuralın yanlış alarmlarının çoğu erken vurulu ya da sinüs aritmili kayıtlarda çıkmaktadır. Bir kardiyoloğun EKG kâğıdında kullandığı hangi ikinci ölçütü kurala eklerdiniz ve bu ölçüt sinyalin hangi bölümüne ihtiyaç duyardı?

**Alıştırma B.** Eşik 0,10'dan 0,15'e yükseltildiğinde kuralın test katındaki duyarlılığı nedir? Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
veri = alistirma_tablosu("B")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma B", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
m = te & has_cv
cevap = round(se_sp_ppv(af[m], cv[m] > 0.15)[0], 3)
teach.check("Alıştırma B", cevap)

## 3. Tek boyutlu bir evrişimli ağ

*Ders: [Tek boyutlu evrişim: Zaman boyunca kayan bir çekirdek](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html#one-dimensional-convolution-a-kernel-that-slides-along-time)*

$k$ örneklik bir çekirdek 12 derivasyon boyunca kayar, altındaki değerleri ağırlıklarıyla çarpar ve toplamlarını öznitelik haritasına yazar. Dersin ağında adımı 2 olan dört katman vardır; böylece sinyalin uzunluğu her katmanda yarıya iner ve 1.000 örnekten 63 örneğe düşer, kanal sayısı ise 12'den 128'e çıkar. Her kanalın zaman boyunca ortalaması son katmana verilir; son katman her etiket için bir skor üretir.

**Formüllerden koda.** $y[n] = b + \sum_{c}\sum_{j} w_{c,j}\, x_c[n + j]$: `nn.Conv1d(12, 32, 7, stride=2, padding=3)`. $P_{conv} = C_{in} \times C_{out} \times k + C_{out}$: `sum(p.numel() for p in layer.parameters())`. Zaman boyunca ortalama `.mean(dim=2)` ile alınır; altı etiketin kaybı ise sigmoid öncesindeki skorlardan hesaplanan ikili çapraz entropi olan `nn.BCEWithLogitsLoss()` ile bulunur.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 3. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB05_zaman_serisi_ve_ekg_yapay_zekasi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(3)

In [ ]:
# Dersteki eğim çekirdeği, vakanın D2 derivasyonunda elle
lead_ii = signals[case, 1, :300]
kernel = np.array([-1.0, 0.0, 1.0])
feature_map = np.array([np.sum(kernel * lead_ii[n - 1:n + 2]) for n in range(1, len(lead_ii) - 1)])
n_max = int(np.argmax(feature_map)) + 1
print("En büyük çıktı şu örnekte:", n_max, "·", lead_ii[n_max - 1:n_max + 2].round(2), "→", round(float(feature_map.max()), 2), "mV")
fig, (a, b) = plt.subplots(2, 1, figsize=(9, 3.6), sharex=True)
a.plot(np.arange(300) / 100, lead_ii)
a.set_ylabel("D2 (mV)")
b.plot(np.arange(1, 299) / 100, feature_map, color="#9B1B1B")
b.set_ylabel("öznitelik haritası")
b.set_xlabel("zaman (s)")
plt.show()

**Python notu.** `lead_ii[n - 1:n + 2]`, `n` örneğinin çevresindeki üç örneği alır; `np.sum(kernel * ...)` bunları üç ağırlıkla çarpar ve çarpımları toplar: Bu, öznitelik haritasının bir değeridir. Liste üreteci bu işlemi her konumda tekrarlar. En büyük çıktı bir QRS kompleksinin en dik çıkışında yer alır. Bir evrişim katmanı aynı işi, her biri ağırlıklarını kendisi öğrenen 32 çekirdekle aynı anda yapar.

In [ ]:
def conv_block(c_in, c_out, k):
    """Bir katman: Zaman boyunca adımı 2 olan bir evrişim, batch normalizasyonu ve ReLU."""
    return [nn.Conv1d(c_in, c_out, k, stride=2, padding=k // 2), nn.BatchNorm1d(c_out), nn.ReLU()]


class RhythmNet(nn.Module):
    """12 derivasyon üzerinde dört evrişim, her kanalın zaman boyunca ortalaması ve her etiket için bir çıktı."""

    def __init__(self, n_labels=6):
        super().__init__()
        self.features = nn.Sequential(*conv_block(12, 32, 7), *conv_block(32, 64, 5),
                                      *conv_block(64, 64, 5), *conv_block(64, 128, 3))
        self.out = nn.Linear(128, n_labels)

    def forward(self, x):
        return self.out(self.features(x).mean(dim=2))


def predict(net, *inputs, batch=256):
    """Etiketlerin olasılıkları; eğitim yapılmadan batch'ler hâlinde hesaplanır."""
    net.eval()
    with torch.no_grad():
        return torch.cat([torch.sigmoid(net(*[x[i:i + batch] for x in inputs]))
                          for i in range(0, len(inputs[0]), batch)]).numpy()

rhythm_net = RhythmNet(len(RHYTHMS))
print("Eğitilebilir parametreler:", sum(p.numel() for p in rhythm_net.parameters()))
h = torch.zeros(1, 12, 1000)
for layer in rhythm_net.features:
    h = layer(h)
    if isinstance(layer, nn.Conv1d):
        print(f"{layer.in_channels:3d} → {layer.out_channels:3d} kanal, çekirdek {layer.kernel_size[0]} · çıktı {tuple(h.shape)}")

**Python notu.** `class RhythmNet(nn.Module)`, 3. ve 4. haftaların ağlarıyla aynı kalıbı izler: `__init__` katmanları oluşturur, `forward` ise bir batch'in bu katmanlardan nasıl geçeceğini söyler. `conv_block` bir düzeyin üç katmanını liste olarak döndürür; `*` dört listeyi açarak `nn.Sequential` içine yerleştirir. `padding=k // 2` her iki uca yarım çekirdek uzunluğunda sıfır ekler; böylece 2 adımı uzunluğu tam olarak yarıya indirir. `predict`, `net.eval()` ile ağı değerlendirme kipine alır ve olasılıkları gradyan hesaplamadan, batch'ler hâlinde bulur.

In [ ]:
x_all = torch.tensor(np.clip(signals, -5, 5))
y_all = torch.tensor(np.array(labels, dtype=np.float32))
x_train, y_train = x_all[tr], y_all[tr]
x_val, x_test = x_all[va], x_all[te]
y_val, y_test = labels.to_numpy()[va], labels.to_numpy()[te]

torch.manual_seed(RANDOM_STATE)
rhythm_net = RhythmNet(len(RHYTHMS))
EPOCHS, BATCH = 15, 64
optimizer = torch.optim.Adam(rhythm_net.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.003, total_steps=EPOCHS * math.ceil(len(x_train) / BATCH))
loss_fn = nn.BCEWithLogitsLoss()
best_auc, best_epoch, val_history = -1.0, 0, []
for epoch in range(1, EPOCHS + 1):
    rhythm_net.train()
    order = torch.randperm(len(x_train))
    for i in range(0, len(order), BATCH):
        b = order[i:i + BATCH]
        optimizer.zero_grad()
        loss = loss_fn(rhythm_net(x_train[b]), y_train[b])
        loss.backward()
        optimizer.step()
        scheduler.step()
    val_auc = macro_auc(y_val, predict(rhythm_net, x_val))
    val_history.append(val_auc)
    if val_auc > best_auc:
        best_auc, best_epoch, best_state = val_auc, epoch, copy.deepcopy(rhythm_net.state_dict())
    print(f"epoch {epoch:2d} · 9. katta ortalama AUC: {val_auc:.3f}")
rhythm_net.load_state_dict(best_state)
val_prob, test_prob = predict(rhythm_net, x_val), predict(rhythm_net, x_test)

fig, ax = plt.subplots(figsize=(5, 2.8))
ax.plot(range(1, EPOCHS + 1), val_history, marker="o")
ax.set_xlabel("epoch")
ax.set_ylabel("ortalama AUC, 9. kat")
plt.show()
print("En iyi epoch:", best_epoch, "· 9. katta ortalama AUC:", round(best_auc, 3))

**Python notu.** Döngü 3. ve 4. haftalardakiyle aynıdır. `torch.randperm` eğitim kayıtlarını her epoch'ta karıştırır, `OneCycleLR` öğrenme hızını 15 epoch boyunca önce 0,003'e çıkarır, sonra yeniden düşürür; en iyi epoch 9. kattaki ortalama AUC'ye göre seçilir. Eğitim bir dizüstü bilgisayarda yaklaşık bir dakika sürer; dersin çalıştırmasında en iyi epoch 12. epoch olmuş ve 9. katta ortalama AUC 0,881'e ulaşmıştır.

**Alıştırma C.** Aynı ağın altı yerine on iki etiket için, yani `RhythmNet(12)` için kaç parametresi vardır?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma C", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = sum(p.numel() for p in RhythmNet(12).parameters())
teach.check("Alıştırma C", cevap)

## 4. Etiket etiket: AUC ve eşikler

*Ders: [Aynı anda birçok etiket: Çok etiketli sınıflandırma](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html#many-labels-at-once-multi-label-classification)*

Çok etiketli bir ağın her etiketi kendi sorusunu cevapladığı için her etiket ayrı değerlendirilir: Test katında bir AUC ve olasılığı bir karara dönüştüren bir eşik. Eşik 9. katta, duyarlılık ve özgüllük toplamının en yüksek olduğu noktada seçilir ve ancak bundan sonra 10. kata uygulanır.

**Formüllerden koda.** $p_k = \sigma(z_k)$: `predict` içindeki `torch.sigmoid(net(x))`. $k$ etiketinin eşiği, ROC eğrisinin eşikleri arasında Youden indeksi olan $\mathrm{Se} + \mathrm{Sp} - 1$ değerini en büyük yapan eşiktir: `youden_threshold` içindeki `thresholds[np.argmax(tpr - fpr)]`. $\mathrm{PPV} = DP/(DP + YP)$: `se_sp_ppv` fonksiyonunun üçüncü değeri.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 4. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB05_zaman_serisi_ve_ekg_yapay_zekasi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(4)

In [ ]:
rows = []
fig, ax = plt.subplots(figsize=(4.6, 4.2))
for k, label in enumerate(RHYTHMS):
    threshold = youden_threshold(y_val[:, k], val_prob[:, k])
    se, sp, ppv = se_sp_ppv(y_test[:, k], test_prob[:, k] >= threshold)
    auc = roc_auc_score(y_test[:, k], test_prob[:, k])
    fpr, tpr, _ = roc_curve(y_test[:, k], test_prob[:, k])
    ax.plot(fpr, tpr, lw=1.2, label=f"{label} {auc:.3f}")
    rows.append({"etiket": label, "pozitif": int(y_test[:, k].sum()), "AUC": round(auc, 3),
                 "eşik": round(threshold, 3), "duyarlılık": round(se, 3), "özgüllük": round(sp, 3), "PPV": round(ppv, 3)})
ax.plot([0, 1], [0, 1], ls="--", color="#999999", lw=0.8)
ax.set_xlabel("1 − özgüllük")
ax.set_ylabel("duyarlılık")
ax.legend(fontsize=8)
plt.show()
label_table = pd.DataFrame(rows)
print(label_table)
print("Test katında ortalama AUC:", round(label_table["AUC"].mean(), 3))

**Python notu.** `roc_curve` yanlış pozitif oranını, doğru pozitif oranını ve bu oranların değiştiği eşikleri döndürür; `test_prob[:, k] >= threshold`, `k` etiketinin olasılıklarını kararlara dönüştürür. `rows.append({...})` her etiket için bir sözlük toplar; `pd.DataFrame(rows)` bunlardan bir tablo oluşturur.

In [ ]:
# Ağ vaka için ne söylüyor?
case_prob = predict(rhythm_net, x_all[case:case + 1])[0]
print(pd.Series(case_prob.round(3), index=RHYTHMS))
print("Kardiyologların ifadeleri:", ecgs.loc[case, "codes"])

**Gözlemle ve düşün.** Seyrek etiketlerin eşikleri 0,5'in çok altında kalır ve pozitif öngörü değerleri düşüktür. Hangi etiket için çok sayıda yanlış alarmı kabul ederdiniz, hangisi için bazı kayıtları kaçırmayı tercih ederdiniz? Kendi kliniğinizde bir alarmdan sonra neler olduğunu düşünün.

**Alıştırma D.** STACH etiketinin olasılığı 9. katta seçilen eşik yerine 0,5'te kesildiğinde, bu etiketin test katındaki pozitif öngörü değeri nedir? Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
veri = alistirma_tablosu("D")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma D", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(se_sp_ppv(y_test[:, 2], test_prob[:, 2] >= 0.5)[2], 3)
teach.check("Alıştırma D", cevap)

## 5. RR aralıkları üzerinde tekrarlayan bir ağ

*Ders: [Tekrarlayan ağlar: Sinyal boyunca ilerleyen bir durum](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html#recurrent-networks-a-state-that-moves-along-the-signal)*

Tekrarlayan bir ağ bir diziyi her seferinde bir eleman okuyarak işler ve adımdan adıma bir durum taşır. Burada dizi bir kaydın 7 ile 32 arasında değişen sayıdaki RR aralığıdır; ağ ise durumu 16 sayıdan oluşan kapılı bir tekrarlayan birimdir (GRU). Ağ EKG'nin dalgalarından hiçbir şey görmez, yalnızca atımların zamanlamasını görür.

**Formüllerden koda.** Güncelleme kapısı $z_t$ olmak üzere $h_t = (1 - z_t)\, h_{t-1} + z_t\, \tilde{h}_t$: `nn.GRU(1, 16, batch_first=True)`. Kayıtların aralık sayıları farklı olduğu için `rr_sequences` her diziyi 32 adıma kadar sıfırla doldurur ve dizinin gerçek uzunluğunu saklar; `pack_padded_sequence` ise GRU'ya gerçek uzunlukta durmasını söyler.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 5. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB05_zaman_serisi_ve_ekg_yapay_zekasi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(5)

In [ ]:
MAX_INTERVALS = 32


def rr_sequences(rr_list):
    """GRU için eşit uzunlukta diziler hâline getirilmiş RR aralıkları, (RR - 0,8) / 0,2 ile ölçeklenmiş, ve her dizinin gerçek uzunluğu."""
    seq = np.zeros((len(rr_list), MAX_INTERVALS, 1), dtype=np.float32)
    lengths = np.ones(len(rr_list), dtype=np.int64)
    for i, rr in enumerate(rr_list):
        rr = np.asarray(rr[:MAX_INTERVALS])
        seq[i, :len(rr), 0] = (rr - 0.8) / 0.2
        lengths[i] = max(1, len(rr))
    return torch.tensor(seq), torch.tensor(lengths)


class RRNet(nn.Module):
    """RR aralıklarını tek tek okuyan 16 sayılık bir GRU ve son durumundan her etiket için bir çıktı."""

    def __init__(self, hidden=16, n_labels=6):
        super().__init__()
        self.gru = nn.GRU(1, hidden, batch_first=True)
        self.out = nn.Linear(hidden, n_labels)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        _, h = self.gru(packed)
        return self.out(h[-1])

seq_all, len_all = rr_sequences(rr_list)
torch.manual_seed(RANDOM_STATE)
rr_net = RRNet(n_labels=len(RHYTHMS))
optimizer = torch.optim.Adam(rr_net.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()
train_index = np.where(tr)[0]
best_auc_rr, best_epoch_rr = -1.0, 0
for epoch in range(1, 61):
    rr_net.train()
    order = train_index[torch.randperm(len(train_index)).numpy()]
    for i in range(0, len(order), 64):
        b = order[i:i + 64]
        optimizer.zero_grad()
        loss = loss_fn(rr_net(seq_all[b], len_all[b]), y_all[b])
        loss.backward()
        optimizer.step()
    val_auc = macro_auc(y_val, predict(rr_net, seq_all[va], len_all[va]))
    if val_auc > best_auc_rr:
        best_auc_rr, best_epoch_rr, best_state_rr = val_auc, epoch, copy.deepcopy(rr_net.state_dict())
rr_net.load_state_dict(best_state_rr)
test_prob_rr = predict(rr_net, seq_all[te], len_all[te])

print("GRU ağının parametreleri:", sum(p.numel() for p in rr_net.parameters()), "· en iyi epoch", best_epoch_rr)
comparison = pd.DataFrame({"kural (DK)": [np.nan, round(roc_auc_score(af[te & has_cv], cv[te & has_cv]), 3), np.nan, np.nan, np.nan, np.nan],
                           "evrişimli ağ": label_table["AUC"].to_numpy(),
                           "GRU": [round(roc_auc_score(y_test[:, k], test_prob_rr[:, k]), 3) for k in range(len(RHYTHMS))]}, index=RHYTHMS)
print(comparison)
print("Ortalama AUC:", comparison.iloc[:, 1:].mean().round(3).to_dict())

**Python notu.** `h[-1]`, her kaydın gerçek son aralığından sonraki son GRU durumudur; `self.out` bu durumun 16 sayısını altı skora dönüştürür. `(rr - 0.8) / 0.2` aralıkları tipik bir değerin çevresinde ölçekler; bu da eğitimi hızlandırır. 4.356 kısa dizi üzerinde 60 epoch yaklaşık yarım dakika sürer. `predict(rr_net, seq_all[va], len_all[va])` ağa iki girdi verir: diziler ve uzunlukları.

**Gözlemle ve düşün.** Yaklaşık bin ağırlığıyla ve tek bir dalgayı bile görmeden GRU, etiketlerin çoğunda evrişimli ağa yetişir ya da onu geçer. Evrişimli ağ hangi etikette üstünlüğünü korur ve bunu EKG'nin hangi bölümü açıklar?

**Alıştırma E.** Tekrarlayan ağın durumu 16 yerine 32 sayı tuttuğunda, yani `RRNet(hidden=32)` için ağın kaç parametresi vardır?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma E", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = sum(p.numel() for p in RRNet(hidden=32).parameters())
teach.check("Alıştırma E", cevap)

## 6. Medyan atım ve J noktası

*Ders: [ST segmentini otomatik ölçmek: Medyan atım ve J noktası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html#measuring-the-st-segment-automatically-the-median-beat-and-the-j-point)*

Bir EKG cihazının yazılımı ölçüm yapmadan önce her derivasyon için temiz bir atım oluşturur. Tek tek atımları R tepelerine göre hizalar ve örnek örnek medyanlarını alır; böylece gürültünün ve tek bir erken vurunun etkisi azalır. QRS başlangıcı ve J noktası on iki derivasyonun eğimlerinden birlikte bulunur; her derivasyonun referans düzeyi ise QRS başlangıcından hemen önceki PR segmentidir.

**Formüllerden koda.** R tepeleri $t_i$ üzerinden $\tilde{x}_\ell(\tau) = \mathrm{medyan}_i\, x_\ell(t_i + \tau)$: `np.median(single, axis=0)`. $\mathrm{ST}_\ell = \tilde{x}_\ell(\tau_J) - \mathrm{PR}_\ell$: `case_beat[:, j_point] - level`; burada `level`, QRS başlangıcından 20 ms önce biten 40 ms'nin ortalamasıdır. `beats.qrs_onset_offset`, başlangıcı ve J noktasını genel eğimden, yani on iki derivasyonun mutlak eğimlerinin toplamından bulur.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 6. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB05_zaman_serisi_ve_ekg_yapay_zekasi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(6)

In [ ]:
# Vakanın medyan atımı, adım adım
before, after = 30, 56                   # 100 Hz'de R tepesinden 0,30 s önce ve 0,55 s sonra
single = np.stack([signals[case][:, p - before:p + after] for p in r_peak_list[case] if p - before >= 0 and p + after <= 1000])
case_beat = np.median(single, axis=0)
onset, j_point = beats.qrs_onset_offset(case_beat)
level = case_beat[:, onset - 6:onset - 2].mean(axis=1)
st_case = case_beat[:, j_point] - level
print(single.shape, "→", case_beat.shape)
print("QRS başlangıcı ve J noktası, R tepesine göre ms:", (onset - before) * 10, (j_point - before) * 10)
print(pd.Series(st_case.round(3), index=beats.LEADS))

**Python notu.** `single` dizisinin boyutu (atım, derivasyon, örnek)'tir; `np.median(single, axis=0)` medyanı ilk eksen, yani atımlar üzerinden, her derivasyon ve örnek için ayrı ayrı alır. `case_beat[:, onset - 6:onset - 2]` başlangıçtan 60 ile 30 ms önceki dört örneği, yani PR segmentinin 40 ms'sini alır; `.mean(axis=1)` bunların her derivasyon için ortalamasını hesaplar. J noktasındaki 12 değerden 12 düzeyden oluşan bir vektörün çıkarılması derivasyon derivasyon yapılır.

In [ ]:
ms = np.arange(-before, after) * 10
v2 = beats.LEADS.index("V2")
fig, ax = plt.subplots(figsize=(6, 3.4))
for s in single:
    ax.plot(ms, s[v2], color="#C8C8C8", lw=0.7)
ax.plot(ms, case_beat[v2], color="#231A1A", lw=1.8, label="V2'nin medyan atımı")
ax.axvspan((onset - 6 - before) * 10, (onset - 3 - before) * 10, color="#B97813", alpha=0.2, label="PR segmenti, 40 ms")
ax.axvline((j_point - before) * 10, color="#9B1B1B", ls="--", label="J noktası")
ax.axhline(level[v2], color="#B97813", lw=0.8)
ax.set_xlabel("R tepesine göre zaman (ms)")
ax.set_ylabel("mV")
ax.legend(fontsize=8)
plt.show()
alarm, meet, groups = beats.stemi_criteria(dict(zip(beats.LEADS, st_case)), "M", 68)
print("Alarm:", alarm, "· derivasyonlar:", meet, "· gruplar:", groups)

**Gözlemle ve düşün.** Vakanın QRS kompleksi 120 ms sürer ve J noktası R tepesinden 50 ms sonradır. Program J noktasını 20 ms erken, yani S dalgasının inişinin içine yerleştirseydi V2'nin ST sapması nasıl değişirdi? Erken ya da geç yerleştirmeden hangisi yanlış alarmlara yol açardı?

**Alıştırma F.** Bazı algoritmalar ST segmentini J noktasından 40 ms sonra ölçer. Vakanın V3 derivasyonunda bu noktadaki ST sapması kaç mV'tur? Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
veri = alistirma_tablosu("F")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma F", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
v3 = beats.LEADS.index("V3")
cevap = round(float(case_beat[v3, j_point + 4] - level[v3]), 3)
teach.check("Alıştırma F", cevap)

## 7. 9. ve 10. katlarda ST sapması ve STEMI alarmı

*Ders: [ST yükselmesi ve çökmesi: Evrensel tanımın ölçütleri](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html#st-elevation-and-depression-the-criteria-of-the-universal-definition)*

Eğitimin alarmı, dördüncü evrensel tanımın ölçütlerini her kaydın medyan atımına uygular: İki ardışık derivasyonda J noktasında en az 1 mm'lik yeni ST yükselmesi; V2 ve V3 için eşikler cinsiyete ve yaşa göre seçilir. Gerçek durum, PTB-XL kardiyologlarının verdiği enfarktüs evresidir: I. ve I-II. evreler akut sayılır.

**Formüllerden koda.** $\ell$ derivasyonunun eşiği `beats.cut_point(lead, sex, age)` ile verilir: V2 ve V3'te 0,2; 0,25 ya da 0,15 mV, diğer derivasyonlarda 0,1 mV. Bir grup, derivasyonlarından en az ikisi eşiğine ulaştığında alarm verir: `beats.stemi_criteria` içindeki `sum(n in meet for n in names) >= 2`. Duyarlılık, özgüllük ve PPV `se_sp_ppv(st_table["acute"], st_table["alarm"])` ile bulunur.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 7. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB05_zaman_serisi_ve_ekg_yapay_zekasi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(7)

In [ ]:
st_rows, median_beats = [], {}
for i in np.where(va | te)[0]:
    beat, n_beats = beats.median_beat(signals[i], peaks=r_peak_list[i])
    median_beats[i] = beat
    st = beats.st_deviation(beat)
    alarm, meet, groups = beats.stemi_criteria(st["st_mv"], ecgs["sex"].iloc[i], ecgs["age"].iloc[i])
    st_rows.append({"ecg_id": int(ecgs["ecg_id"].iloc[i]), "split": split[i], "sex": ecgs["sex"].iloc[i], "age": ecgs["age"].iloc[i],
                    "acute": bool(ecgs["acute_mi"].iloc[i]), **st["st_mv"], "alarm": alarm, "groups": " ".join(groups)})
st_table = pd.DataFrame(st_rows)
for name, part in (("9. ve 10. katlar", st_table), ("test", st_table[st_table["split"] == "test"])):
    se, sp, ppv = se_sp_ppv(part["acute"], part["alarm"])
    print(f"{name}: {len(part)} kayıt · {int(part['acute'].sum())} akut enfarktüs · {int(part['alarm'].sum())} alarm"
          f" · duyarlılık {se:.3f} · özgüllük {sp:.3f} · PPV {ppv:.3f}")

**Python notu.** Döngü 9. ve 10. katlardaki her kaydın medyan atımını bir kez oluşturur; bu yaklaşık yarım dakika sürer. Atımlar, 8. bölümdeki QT için `median_beats` sözlüğünde saklanır. `**st["st_mv"]` on iki ST sapmasının sözlüğünü satırın içine açar; böylece her derivasyon `st_table` tablosunun bir sütunu olur.

In [ ]:
codes_of = ecgs.set_index("ecg_id")["codes"]
false_alarm = st_table["alarm"] & ~st_table["acute"]
counts = Counter(c for e in st_table.loc[false_alarm, "ecg_id"] for c in codes_of[e])
print("Yanlış alarmlar:", int(false_alarm.sum()))
print(pd.Series(dict(counts.most_common(12))))
for code_name in ("CLBBB", "PACE", "LVH", "NORM"):
    has_code = st_table["ecg_id"].map(lambda e, c=code_name: c in codes_of[e])
    print(f"{code_name}: {int(st_table.loc[has_code, 'alarm'].sum())} alarm,", int(has_code.sum()), "kayıt")
missed = st_table[st_table["acute"] & ~st_table["alarm"]]
print("Kaçırılan akut enfarktüsler:", [(e, sorted(codes_of[e])) for e in missed["ecg_id"]])

**Gözlemle ve düşün.** Tam LBBB olan kayıtların neredeyse hepsi, kalp pili olan kayıtların da çoğu alarm verir. Evrensel tanım, ST ölçütlerinin bu kayıtlara uygulanamayacağını söyler. Otomatik bir alarm böyle bir kayıtta ne yapmalıdır: sessiz mi kalmalı, bir hekimin okumasını mı istemeli, yoksa başka ölçütler mi uygulamalı?

In [ ]:
def alarm_with_margin(row, margin):
    """Her eşiği margin kadar (mV) yükseltilmiş evrensel tanım alarmı."""
    meet = [lead for lead in beats.LEADS if row[lead] >= beats.cut_point(lead, row["sex"], row["age"]) + margin - 1e-9]
    return any(sum(n in meet for n in names) >= 2 for names in beats.GROUPS.values())


for margin in (0.0, 0.05, 0.10):
    alarm_m = st_table.apply(alarm_with_margin, axis=1, margin=margin)
    se, sp, ppv = se_sp_ppv(st_table["acute"], alarm_m)
    print(f"+{margin:.2f} mV: {int(alarm_m.sum())} alarm · duyarlılık {se:.3f} · PPV {ppv:.3f}")

**Alıştırma G.** Her eşik 0,05 mV yükseltildiğinde alarm 9. ve 10. katlarda kaç alarm verir?

In [ ]:
veri = alistirma_tablosu("G")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma G", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int(st_table.apply(alarm_with_margin, axis=1, margin=0.05).sum())
teach.check("Alıştırma G", cevap)

## 8. Medyan atımda QT aralığı

*Ders: [QT uzaması ve riski](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html#qt-prolongation-and-its-risk)*

QT aralığı QRS başlangıcından T dalgasının sonuna kadar uzanır. Eğitimde başlangıç bütün derivasyonlardan, T dalgasının sonu ise V5'ten teğet yöntemiyle alınır: T dalgasının inen kolunun en dik noktasındaki teğet, taban çizgisini dalganın sonunda keser. QT aralığı daha sonra kalp hızına göre düzeltilir.

**Formüllerden koda.** RR saniye cinsinden olmak üzere $\mathrm{QTc}_B = \mathrm{QT}/\sqrt{RR}$ ve $\mathrm{QTc}_F = \mathrm{QT}/\sqrt[3]{RR}$: `beats.qtc(qt, rr, "Bazett")` ve `beats.qtc(qt, rr, "Fridericia")`. `beats.qt_interval(beat, rr_s=...)` QT aralığını bir medyan atımda ms cinsinden ölçer; T dalgasının sonu bulunamazsa *sayı değil* (not a number) döndürür.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 8. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB05_zaman_serisi_ve_ekg_yapay_zekasi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(8)

In [ ]:
qt_rows = []
for i, beat in median_beats.items():
    rr_mean = float(rr_list[i].mean()) if len(rr_list[i]) else np.nan
    qt_rows.append({"ecg_id": int(ecgs["ecg_id"].iloc[i]), "qt_ms": beats.qt_interval(beat, rr_s=rr_mean), "rr_s": rr_mean,
                    "long_qt": "LNGQT" in ecgs["codes"].iloc[i]})
qt_table = pd.DataFrame(qt_rows)
qt_table["qtc_b"] = beats.qtc(qt_table["qt_ms"], qt_table["rr_s"], "Bazett")
qt_table["qtc_f"] = beats.qtc(qt_table["qt_ms"], qt_table["rr_s"], "Fridericia")
print("QT aralığı bulunamayan kayıtlar:", int(qt_table["qt_ms"].isna().sum()))
print(qt_table.groupby("long_qt")[["qtc_b", "qtc_f"]].median().round(0))
print("500 ms ve üstü QTc (Bazett, Fridericia):", int((qt_table["qtc_b"] >= 500).sum()), int((qt_table["qtc_f"] >= 500).sum()))
fig, ax = plt.subplots(figsize=(6, 3))
bins = np.arange(300, 650, 10)
ax.hist(qt_table.loc[~qt_table["long_qt"], "qtc_b"].dropna(), bins=bins, color="#1F5F8B", alpha=0.6, label="diğer kayıtlar")
ax.hist(qt_table.loc[qt_table["long_qt"], "qtc_b"].dropna(), bins=bins, color="#9B1B1B", alpha=0.8, label="uzun QT ifadesi")
ax.axvline(500, color="k", ls="--", lw=1)
ax.set_yscale("log")
ax.set_xlabel("Bazett formülüyle QTc (ms)")
ax.legend(fontsize=8)
plt.show()

**Python notu.** `median_beats.items()`, 7. bölümün sakladığı kayıt ve medyan atım çiftlerini verir; böylece hiçbir atım iki kez oluşturulmaz. `beats.qtc(qt_table["qt_ms"], qt_table["rr_s"], ...)` bütün sütunlar üzerinde aynı anda çalışır. *Sayı değil* ile yapılan bir karşılaştırma, örneğin `qtc_b >= 500`, `False` verir; bu yüzden bulunamayan bir QT aralığı hiçbir zaman uzun sayılmaz.

**Gözlemle ve düşün.** Bazett formülüyle QTc'si 500 ms ve üstünde olan kayıtların çoğu uzun QT ifadesi taşımaz. Bunlardan hangilerini önce elle kontrol ederdiniz: kalp hızı yüksek olanları mı, QRS kompleksi geniş olanları mı, yoksa V5'te T dalgası düz olanları mı? Neden?

**Alıştırma H.** 9. ve 10. katlardaki kayıtlardan kaçının Fridericia formülüyle QTc'si 500 ms ya da daha fazladır?

In [ ]:
veri = alistirma_tablosu("H")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma H", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int((qt_table["qtc_f"] >= 500).sum())
teach.check("Alıştırma H", cevap)

## 9. Uzun süreli izlem: Sanal klinikte bir gün

*Ders: [Uzun süreli izlem: On saniyeden yirmi dört saate](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html#long-term-monitoring-from-ten-seconds-to-twenty-four-hours)*

Sanal kliniğin bir Holter cihazı VC-0128 numaralı hastayı saat 08.00'den başlayarak 24 saat izlemiştir. Kayıt sentetiktir: Üç paroksismal AF atağı, atriyal erken vuruların sık görüldüğü bir saat, iki elektrot gürültüsü dönemi ve kalp hızının gün boyunca yavaşça artıp azalması kayıtta yer alır. Cihazın atım algılayıcısı, gerçek algılayıcıların gürültüde yaptığı gibi, bazı atımları kaçırır ve bazılarını olmadığı hâlde bulur. Gerçek durum atım atım bilindiği için her alarm denetlenebilir.

**Formüllerden koda.** $L$ saniyelik pencerelerden alarm verilenlerin sayısı $n_{AF}$ olmak üzere $\text{AF yükü} = \frac{n_{AF} \times L}{24 \times 3600}$: `100 * alarms * length / (24 * 3600)`. Bir atımın penceresi `time // length`, yani bölmenin tam kısmıdır; `groupby` her pencerenin RR aralıklarını toplar.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 9. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB05_zaman_serisi_ve_ekg_yapay_zekasi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(9)

In [ ]:
holter_rec = holter.recording()
b = holter_rec["detected_s"]
truth_af = np.zeros(len(b), dtype=int)
for s, e in holter_rec["episodes"]:
    truth_af[(b >= s) & (b <= e)] = 1
holter_beats = pd.DataFrame({"time": b[1:], "RR": np.diff(b), "AF": truth_af[1:]})
print(len(holter_beats), "atım · gerçek AF yükü:", round(100 * holter.burden(holter_rec), 1), "%")
for s, e in holter_rec["episodes"]:
    print("AF:", holter.clock_text(s), "–", holter.clock_text(e))
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.plot(holter_beats["time"] / 3600, holter_beats["RR"], ",", color="#1F5F8B")
for s, e in holter_rec["episodes"]:
    ax.axvspan(s / 3600, e / 3600, color="#B97813", alpha=0.25)
ax.set_xticks(range(0, 25, 4))
ax.set_xticklabels([f"{(8 + h) % 24:02d}:00" for h in range(0, 25, 4)])
ax.set_ylim(0, 2)
ax.set_ylabel("RR (s)")
plt.show()

**Python notu.** `holter.recording()` bir sözlük döndürür; `holter_rec["detected_s"]` atımların algılayıcının bulduğu zamanlarını, `holter_rec["episodes"]` ise her AF atağının başlangıcını ve sonunu saniye olarak tutar. `(b >= s) & (b <= e)` bir atağın içindeki atımları işaretler. Grafik her biri bir nokta olan 100.000'den fazla aralığı gösterir (`","` tek bir piksel çizer).

In [ ]:
# Günden dört D2 derivasyonu şeridi
fig, axes = plt.subplots(4, 1, figsize=(10, 6.4), sharex=True)
for ax, (clock, title) in zip(axes, [("08:30", "sinüs ritmi"), ("13:25", "AF"),
                                     ("18:30", "atriyal erken vurular"), ("07:05", "elektrot gürültüsü")]):
    start = ((int(clock[:2]) + int(clock[3:]) / 60 - 8) % 24) * 3600
    t, strip = holter.strip(holter_rec, start, 10)
    teach.plot_strip(ax, t, strip, label=f"{clock} · {title}", v_range=(-1.5, 2.5))
plt.show()

In [ ]:
def window_table(beats_df, length):
    """length saniyelik pencereler: RR aralıklarının değişim katsayısı, gerçek durum ve kalite kuralı."""
    w = (beats_df["time"] // length).astype(int)
    g = beats_df.groupby(w)
    out = pd.DataFrame({"cv": g["RR"].std() / g["RR"].mean(), "af": g["AF"].mean() > 0.5,
                        "bad": g["RR"].apply(lambda r: bool(((r < 0.3) | (r > 1.6)).any()))})
    out["hour"] = ((8 + out.index * length / 3600) % 24).astype(int)
    return out


rows = []
for length in (30, 60, 120):
    wt = window_table(holter_beats, length)
    for quality in (False, True):
        use = ~wt["bad"] if quality else pd.Series(True, index=wt.index)
        alarm = (wt["cv"] > 0.10) & use
        rows.append({"pencere (s)": length, "kalite kuralı": quality, "alarm": int(alarm.sum()),
                     "yanlış": int((alarm & ~wt["af"]).sum()), "kaçırılan": int((~alarm & wt["af"]).sum()),
                     "yük (%)": round(100 * alarm.sum() * length / (24 * 3600), 1)})
print(pd.DataFrame(rows))
wt30 = window_table(holter_beats, 30)
false_30 = (wt30["cv"] > 0.10) & ~wt30["af"]
print("30 saniyelik pencerelerde saatlere göre yanlış alarmlar:", wt30[false_30].groupby("hour").size().to_dict())

**Python notu.** `window_table` her pencere için bir satır verir. `g["RR"].apply(lambda r: ...)` her pencerenin aralıkları üzerinde küçük bir fonksiyon çalıştırır: `lambda r:` adı olmayan bir fonksiyon tanımlar; buradaki fonksiyon, herhangi bir aralığın 0,3 saniyeden kısa ya da 1,6 saniyeden uzun olup olmadığını sorar. `use` bütün pencereleri, kalite kuralı açıkken ise yalnızca kötü olmayan pencereleri tutar; dışarıda bırakılan bir pencere alarm veremez, bu yüzden kuralın dışarıda bıraktığı bir AF penceresi kaçırılmış sayılır. Gerçek yük yüzde 15,4'tür; her satırı bu değerle karşılaştırın.

**Gözlemle ve düşün.** 120 saniyelik pencereler ve kalite kuralıyla hiç yanlış alarm kalmaz, ama kestirilen yük gerçek değerin altına düşer. Kural hangi AF pencerelerini dışarıda bırakır ve neden? Bu pencereleri tutması için kuralda neyi değiştirirdiniz?

**Alıştırma I.** 60 saniyelik pencereler ve kalite kuralıyla günün AF yükü yüzde kaçtır? Sonucu bir ondalık basamakla verin.

In [ ]:
veri = alistirma_tablosu("I")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma I", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
wt60 = window_table(holter_beats, 60)
cevap = round(100 * int(((wt60["cv"] > 0.10) & ~wt60["bad"]).sum()) * 60 / (24 * 3600), 1)
teach.check("Alıştırma I", cevap)

## 10. YZ asistanıyla çalışmak, 5. bölüm: Bir zaman serisi işlem hattını denetlemek

*Ders: [YZ asistanıyla çalışmak, 5. bölüm: Bir zaman serisi işlem hattını denetlemek](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html#ai-pair-programming-part-5-reviewing-a-time-series-pipeline)*

Bir katılımcı, bir YZ asistanından *PTB-XL'in D2 derivasyonundaki RR aralıklarından bir AF tespit modeli eğitmesini ve modelin ne kadar iyi olduğunu bildirmesini* istemiştir. Aşağıdaki kod asistanın cevabıdır. Kod çalışır ve iyi görünen bir doğruluk yazdırır. Ona güvenmeden önce kodu zaman serisi işlem hattı denetiminin altı sorusuyla okuyun: Bir hastanın kayıtları tek bir kümede mi tutulmuş? Örnekleme frekansı ve birimler doğru mu? Derivasyonlar adlarıyla mı alınmış? Eşikler doğrulama katında mı seçilmiş? Sonuçlar her etiket için prevalansıyla birlikte mi bildirilmiş? Ölçümün kuralı belirtilmiş mi?

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 10. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB05_zaman_serisi_ve_ekg_yapay_zekasi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(10)

In [ ]:
# YZ asistanının cevabı
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

lead_ii_all = signals[:, 0]                          # lead II
peaks_ai = [beats.r_peaks(s, fs=500) for s in lead_ii_all]
rr_ai = [np.diff(p) / 500 for p in peaks_ai]
features_ai = np.array([[np.mean(r), np.std(r)] if len(r) > 1 else [0.0, 0.0] for r in rr_ai])
y_ai = labels["AFIB"].to_numpy()
f_train, f_test, y_train_ai, y_test_ai = train_test_split(features_ai, y_ai, test_size=0.3, random_state=0)
model_ai = LogisticRegression().fit(f_train, y_train_ai)
print("Accuracy:", round(model_ai.score(f_test, y_test_ai), 3))
prob_ai = model_ai.predict_proba(f_test)[:, 1]
fpr_ai, tpr_ai, thr_ai = roc_curve(y_test_ai, prob_ai)
k_ai = int(np.argmax(tpr_ai - fpr_ai))
print("Best threshold:", round(float(thr_ai[k_ai]), 3), "· sensitivity:", round(float(tpr_ai[k_ai]), 3))

Kodun dört sorunu ve bir eksiği vardır. `signals[:, 0]` D2 değil D1 derivasyonudur: PTB-XL'de derivasyonlar D1, D2, D3 sırasıyla saklanır ve bir derivasyon adıyla, yani `beats.LEADS.index("II")` ile alınmalıdır. `fs=500` algılayıcıya sinyalin saniyede 500 örnek içerdiğini söyler; oysa eğitimin kayıtlarında saniyede 100 örnek vardır. Algılayıcının filtreleri ve refrakter süresi yanlış ölçekte çalışır ve her aralık beş kat kısa çıkar. `train_test_split` bir hastanın kayıtlarını eğitim ve test arasında karıştırır ve PTB-XL'in katlarını yok sayar. Eşik de test kümesinin kendisinde seçilmiştir. Eksik olan şudur: Doğruluk, AF'nin ne kadar seyrek olduğunu gizler. Hiç alarm vermeyen bir program bu kayıtlarda 0,93 doğruluğa ulaşır; çünkü kayıtların yüzde 93'ünde AF yoktur.

**Alıştırma J.** Hattı düzeltin: 2. bölümde on iki derivasyonun hepsinden 100 Hz'de bulunan RR aralıklarını ve aynı iki özniteliği, yani ortalamayı ve `np.std` ile standart sapmayı kullanın. Standartlaştırılmış özniteliklerle 1. ve 2. katlarda eğitin, eşiği 9. katta seçin ve 10. kattaki duyarlılığı üç ondalık basamağa yuvarlayarak bildirin.

In [ ]:
veri = alistirma_tablosu("J")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma J", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
features_rr = np.array([[np.mean(r), np.std(r)] if len(r) > 1 else [0.0, 0.0] for r in rr_list])
model_rr = make_pipeline(StandardScaler(), LogisticRegression()).fit(features_rr[tr], af[tr])
threshold_rr = youden_threshold(af[va], model_rr.predict_proba(features_rr[va])[:, 1])
cevap = round(se_sp_ppv(af[te], model_rr.predict_proba(features_rr[te])[:, 1] >= threshold_rr)[0], 3)
teach.check("Alıştırma J", cevap)

Etkileşimli labdaki [prompt atölyesinde](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/lab.html#part-f) böyle bir işlem hattının promptu, prompt kartının yedi alanıyla yazılır. *Veri* alanı örnekleme frekansını, derivasyonların sırasını ve katları belirtir; *Test* alanı ise doğruluğun yanında duyarlılığı ve pozitif öngörü değerini de ister.

## Alanlara göre uygulama görevleri

| Alan | Görev |
|---|---|
| Elektrofizyoloji | 2. bölümdeki kural da 5. bölümdeki GRU da yalnızca RR aralıklarını okur. Kuralın 9. kattaki özgüllüğü GRU'nun AF eşiğindeki özgüllüğüne eşit olana kadar kuralın eşiğini yükseltin ve iki duyarlılığı 10. katta karşılaştırın. Sık erken vurusu olan bir hastada hangisine güvenirdiniz? |
| Girişimsel kardiyoloji | 7. bölümdeki alarmın kaçırdığı akut enfarktüslerin medyan atımlarına bakın. Bunlardan kaçı V1-V3'te ST çökmesi gösteriyor ve posterior derivasyonlar (V7-V9) alarma ne katardı? |
| Göğüs ağrısı birimi | 7. bölümdeki pozitif öngörü değeriyle bir gecede verilen 50 alarmdan kaçı doğru olurdu? Kateter ekibi çağrılmadan önce bir birimin bir alarmla ne yapması gerektiğini yazın. |
| Kalp yetersizliği | Sol dal bloğu kalp yetersizliğinde sık görülür. 7. bölümdeki alarmların ne kadarının tam LBBB olan kayıtlardan geldiğini hesaplayın ve bu kayıtlar için bir kural önerin; örneğin Sgarbossa ölçütleri. |
| Yoğun bakım | Yoğun bakımda kullanılan birçok ilaç QT aralığını uzatır. 8. bölümdeki `qt_table` tablosunu kullanın: Kaç kayıt Bazett formülüyle 500 ms'ye ulaşır ama Fridericia formülüyle ulaşmaz ve bu kayıtların kalp hızları nedir? |
| Kardiyo-onkoloji | Bir hastanın QTc'si QT aralığını uzatan bir ilaç başlanmadan önce 470 ms, başlandıktan sonra 505 ms'dir. Eğitimin ölçümünün sanal klinikteki hatası, yani yaklaşık 16 ms'lik standart sapması göz önüne alındığında, ne büyüklükteki bir değişikliğe gerçek derdiniz? |
| Ritim polikliniği | 9. bölümdeki kalite kuralını, hızlı AF pencerelerini tutacak ve gürültülü dönemlerin yanlış alarmlarını yine ayıklayacak biçimde değiştirin. Gürültünün hangi özelliğini kullandınız ve AF yükü şimdi kaçtır? |

## Defterden sonra

[Etkileşimli labı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/lab.html) açın ve *Kendini sına* sekmesindeki öz değerlendirmeyi yapın. Ardından *Düşün ve dışa aktar* sekmesinde yansıtma yazınızı yazın ve öğrenme günlüğünü dışa aktarın. [Hafta sayfasında](README.md) anlatılan 5. haftanın haftalık projesinde PTB-XL'in 9. ve 10. katlarında otomatik bir STEMI alarmı kurulur ve bu alarm bir göğüs ağrısı biriminin yapacağı gibi değerlendirilir.

<div align="center">

---

**Prof. Dr. Utku Köse**<br>
Profesör, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi, Isparta, Türkiye<br>
Kurucu Müdür, Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM), Süleyman Demirel Üniversitesi<br>
Bilgisayar Bilimleri Anabilim Dalı Başkanı, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi<br>
Diğer kurumlar: University of North Dakota (ABD), Universidad Panamericana (Meksiko, Meksika), Vel Tech University (Chennai, Hindistan)<br>
IEEE Kıdemli Üyesi, ACM Profesyonel Üyesi

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı, Türk Kardiyoloji Derneği. İlk sürüm: Kasım 2026. İçerik CC BY 4.0, kod MIT.*

</div>